# Heard Island: ice, terrain & change

Build the island explorer, glacier dashboards and 3D terrain views from the project's existing outputs.

**Run this notebook from the top with the KGG375_AT3 environment.** Keep it inside the project folder, or set `PROJECT_FOLDER` below. It replaces the earlier incremental map build; do not paste these cells into the old notebook. The export is written to `Output_Data/processed/interactive_map_data/explorer/`.

The notebook generates a portable HTML **and assets folder**. A small iframe previews that export at the end. Browser data and images are loaded when needed instead of being duplicated inside notebook output. The preview uses a loopback server while the kernel is running; the exported folder and ZIP work without Python. If your notebook frontend blocks a local iframe, use the full-window link or open the HTML. For a remote kernel, forward the printed preview port.

The main view keeps island context, the five dated inventories and selected terrain/flow products. Glacier dashboards contain the additional bed products, quality diagnostics and field evidence. Recent outlines for glaciers other than Brown and Stephenson remain **provisional**. Missing files and dates are reported rather than replaced.

**Scientific boundary:** retreat animations show mapped footprints on fixed terrain. They do not reconstruct historical ice-surface heights. Modelled bed, DEM surface, lagoon depths and velocity products retain their distinct dates, support and vertical-reference limitations.

## Set up the build

All configuration and imports run here. Bounded raster reads and one-glacier-at-a-time export limit memory use. The original analysis files are left intact.

In [ ]:
## Set paths and a bounded display budget ##

from pathlib import Path
from datetime import datetime, timezone
from functools import partial
from html import escape
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread
from urllib.request import urlopen, Request
import gc
import faulthandler
import hashlib
import importlib.metadata
import json
import math
import shutil
import sys
import warnings
import zipfile

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.features import geometry_mask, rasterize
from rasterio.transform import from_bounds
from rasterio.vrt import WarpedVRT
from rasterio.warp import calculate_default_transform, transform_bounds, transform_geom
from shapely.geometry import LineString, mapping, shape, box
from shapely.ops import unary_union
from shapely import force_2d, make_valid
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize, BoundaryNorm
from PIL import Image
from IPython.display import display, HTML, IFrame

# Usually leave this as None. If needed, set the folder containing Input_Data and Output_Data.
PROJECT_FOLDER = None
MAX_ISLAND_IMAGE_SIDE = 1200
MAX_GLACIER_IMAGE_SIDE = 750
MAX_ISLAND_3D_SIDE = 160
MAX_GLACIER_3D_SIDE = 125
DASHBOARD_BUFFER_M = 600
MAKE_PORTABLE_ZIP = True

project_dir = (Path(PROJECT_FOLDER).expanduser().resolve() if PROJECT_FOLDER else
    next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "Output_Data" / "processed").is_dir()), None))
if project_dir is None:
    raise FileNotFoundError("Set PROJECT_FOLDER to the folder containing Output_Data, then Run All.")
input_dir = project_dir / "Input_Data"
processed_dir = project_dir / "Output_Data" / "processed"
map_data_dir = processed_dir / "interactive_map_data"
synthesis_dir = processed_dir / "synthesis"
explorer_dir = map_data_dir / "explorer"
asset_dir = explorer_dir / "assets"
raster_cache_dir = asset_dir / "rasters"
for folder in [explorer_dir, asset_dir, raster_cache_dir, asset_dir / "data", asset_dir / "vendor"]:
    folder.mkdir(parents = True, exist_ok = True)

build_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
# Capture native crash tracebacks when Python can handle the signal.
# An operating-system out-of-memory kill cannot always be logged this way.
if "explorer_crash_handle" in globals():
    faulthandler.disable()
    explorer_crash_handle.close()
explorer_crash_path = explorer_dir / f"explorer_crash_{build_id}.log"
explorer_crash_handle = explorer_crash_path.open("w", buffering = 1)
faulthandler.enable(file = explorer_crash_handle, all_threads = True)
build_issues = []
focus_names = ["Brown", "Stephenson"]
map_colours = {"Brown": "#FFB36B", "Stephenson": "#69D5FF"}
years = [1947, 1988, 2014, 2019, 2026]
year_colours = {1947: "#F4DF70", 1988: "#BE9AFF", 2014: "#E8EFF5", 2019: "#FF83AF", 2026: "#8FE88C"}


def read_json(path):
    return json.loads(Path(path).read_text(encoding = "utf-8-sig"))


def project_path(value):
    """Rebase saved Windows paths onto the current project folder."""
    parts = str(value).replace("\\", "/").split("/")
    for anchor in ["Input_Data", "Output_Data"]:
        if anchor in parts:
            return project_dir.joinpath(*parts[parts.index(anchor):])
    path = Path(value)
    return path if path.is_absolute() else project_dir / path


def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii = False, allow_nan = False,
        separators = (",", ":")), encoding = "utf-8")


def write_data_script(path, variable, value):
    # Script assets also work when the exported HTML is opened directly from disk.
    content = json.dumps(value, ensure_ascii = False, allow_nan = False,
        separators = (",", ":")).replace("</", "<\\/")
    Path(path).write_text(f"{variable} = {content};\n", encoding = "utf-8")


metadata_paths = {
    "animation": map_data_dir / "animation_settings.json",
    "velocity": map_data_dir / "velocity_layers.json",
    "field": map_data_dir / "field_reconstruction_layers.json",
    "bed": processed_dir / "geomorphology" / "bed_estimate_metadata.json"}
required = [map_data_dir / "glacier_inventories.geojson", *metadata_paths.values()]
missing = [str(p.relative_to(project_dir)) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Required outputs are missing:\n" + "\n".join(missing))
map_metadata = {name: read_json(path) for name, path in metadata_paths.items()}
inventory_geojson = read_json(map_data_dir / "glacier_inventories.geojson")
outlines = gpd.GeoDataFrame.from_features(inventory_geojson["features"], crs = "EPSG:4326").to_crs("EPSG:32743")
outlines.geometry = outlines.geometry.map(lambda g: make_valid(force_2d(g)))
glacier_catalogue = outlines.loc[outlines.year.eq(2014), ["RGIId", "name_1"]].drop_duplicates().sort_values("name_1")
print(f"Project: {project_dir}\nGlaciers: {len(glacier_catalogue)}\nExport: {explorer_dir}")
print("Images are bounded display copies. The original analytical datasets are not changed.")

## Record sources, metadata and credits

Dataset credits are exported alongside processing metadata and linked from every layer. Source details that cannot be established from the supplied files remain explicitly unresolved. The accepted RADARSAT DEM has its own citation.

In [ ]:
## Start the source register used by the credits pop-up ##

source_register = {}


def register_source(source_id, title, citation = None, url = None,
        licence = None, **details):
    # Keep missing citation details explicit until their source is verified.
    source_register[source_id] = {
        "id": source_id, "title": title, "citation": citation,
        "url": url, "licence": licence, "metadata": details,
        "citation_status": "verified" if citation else "needs_verification"}


register_source("dem_2002", "Heard Island RADARSAT (2002) DEM",
    citation = (
        "Brolsma, H. & Smith, D. T. (2008). Heard Island RADARSAT (2002) "
        "Digital Elevation Model (DEM), Version 1. Australian Antarctic Data Centre."),
    url = "https://data.aad.gov.au/metadata/records/heard_dem_radarsat02",
    licence = "CC BY 4.0; retain the source record's access conditions",
    observation_period = "2002, merged with a 1997 stereoscopic DEM",
    original_cell_size_m = 10,
    working_cell_size_m = 50,
    vertical_reference = "Source vertical reference must be checked against the supplied DEM documentation; do not equate lagoon survey depths with this datum.",
    local_metadata = "Input_Data/Heard_DEM_2002/heard_dem_radarsat02.xml")

register_source("thickness", "Global glacier ice thickness",
    citation = (
        "Millan, R., Mouginot, J. & Rabatel, A. (2021). "
        "Ice velocity and thickness of the world's glaciers [Dataset]. Theia."),
    url = "https://doi.org/10.6096/1007",
    doi = "10.6096/1007",
    representative_period = "2010–2020",
    publication = "https://doi.org/10.1038/s41561-021-00885-z",
    processing = map_metadata["bed"])

register_source("its_live", "ITS_LIVE Sentinel-1 image-pair velocities, Version 2",
    citation = (
        "Lei, Y., Gardner, A. S., Kennedy, J., Fahnestock, M., Agram, P., "
        "Liukis, M., Greene, C., Johnston, A., Lopez, L. & Scambos, T. (2022). "
        "MEaSUREs ITS_LIVE Sentinel-1 Image-Pair Glacier and Ice Sheet Surface "
        "Velocities, Version 2. National Snow and Ice Data Center."),
    url = "https://doi.org/10.5067/0506KQLS6512",
    doi = "10.5067/0506KQLS6512",
    grid_spacing_m = 120,
    selected_data = map_metadata["velocity"]["sources"],
    selection = map_metadata["velocity"]["settings"],
    method_reference = "https://doi.org/10.5194/essd-14-5111-2022")

register_source("geology", "Heard Island Geology Map",
    citation = (
        "Fox, J., Carey, R. J. & McPhie, J. (2023). Heard Island Geology Map – "
        "compiled from data collected from 1929–2020, Version 1. "
        "Australian Antarctic Data Centre."),
    url = "https://doi.org/10.26179/nb02-vj63",
    licence = "CC BY 4.0",
    doi = "10.26179/nb02-vj63",
    observation_period = "1929–2020",
    additional_credit = (
        "Fox et al. (2021), Construction of an intraplate island volcano: "
        "The volcanic history of Heard Island."))

register_source("field_2363", "Heard Island glacier fluctuations: 2003/04 fieldwork",
    citation = (
        "Allison, I. & Thost, D. E. (2010). Heard Island glacier fluctuations "
        "and climatic change – 2003/04 Fieldwork, Version 1. "
        "Australian Antarctic Data Centre."),
    url = "https://doi.org/10.4225/15/574BBEA0D74B7",
    licence = "CC BY 4.0",
    doi = "10.4225/15/574BBEA0D74B7",
    observation_period = (
        "2003–2004; archive also includes some November 2000 data"),
    local_metadata = (
        "Input_Data/ASAC2363_Brown_Stephenson_Field_Data/ASAC_2363.xml"))

# These inputs require their own source records, including layers used as masks.
for source_id, title in {
        "inventory_2014": "2014 glacier inventory: Glacier_2014.shp",
        "historical_outlines": "Supplied historical glacier outlines, 1947 and 1988",
        "sentinel_1": "Sentinel-1 imagery and coherence used for recent outlines",
        "coastline": "HIMI coastline polygon: himi_coastline_py.gpkg",
        "lagoons": "Updated 2014 lagoon polygons",
        "campaign_archive": "Brown campaign archive: 2003-04_CampaignData"}.items():
    register_source(source_id, title)

# Preserve processing provenance alongside the bibliographic records.
provenance_register = {
    name: {
        "metadata_file": path.relative_to(project_dir).as_posix(),
        "content": map_metadata[name]}
    for name, path in metadata_paths.items()}


# Every input retains a credit, even where a supplied derivative lacks a full citation.
register_source("project_analysis", "Heard Island project analyses",
    citation = "KGG375 AT3 Group 2 (2026). Heard Island geospatial analysis [Derived datasets].",
    processing_notebooks = [f"{n:02d}" for n in range(1, 10)])
source_register["coastline"].update(
    citation = "Australian Antarctic Data Centre. HIMI coastline GIS layer (supplied polygon export).",
    url = "https://www.antarctica.gov.au/antarctic-operations/stations-and-field-locations/heard-island/mapping/",
    citation_status = "provider_verified; exact export date not recorded")
source_register["coastline"]["metadata"].update(
    display_processing = "Polygon boundary extracted for line display; original polygon retained for masks.")
source_register["sentinel_1"].update(
    citation = "European Union / Copernicus Sentinel-1. Imagery used in project-derived 2019 and 2026 glacier outlines.",
    url = "https://sentiwiki.copernicus.eu/web/s1-mission",
    citation_status = "provider_verified; scene identifiers retained in project provenance")
for key, note in {
        "inventory_2014": "Supplied Glacier_2014.shp. Stable RGI identifiers do not establish the precise source/version of this edited file.",
        "historical_outlines": "Supplied 1947 and 1988 outlines. The 1947 layer is nominal and includes some earlier observations.",
        "lagoons": "Supplied UpdatedHI_Lagoons2014.shp. This is a polygon extent, not a water-level measurement.",
        "campaign_archive": "Supplied 2003-04_CampaignData archive. Original authorship, release date and licence need confirmation from the archive."}.items():
    source_register[key]["metadata"]["provenance_note"] = note
    source_register[key]["citation"] = note
    source_register[key]["citation_status"] = "supplied_input; bibliographic details unresolved"

imagery_url = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer"
imagery_credit = "Source: Esri, Vantor, Earthstar Geographics, and the GIS User Community"
metadata_cache = explorer_dir / "esri_imagery_metadata.json"
try:
    if metadata_cache.is_file():
        imagery_metadata = read_json(metadata_cache)
    else:
        with urlopen(imagery_url + "?f=pjson", timeout = 8) as response:
            imagery_metadata = json.load(response)
        write_json(metadata_cache, imagery_metadata)
    imagery_credit = imagery_metadata.get("copyrightText") or imagery_credit
except Exception as exc:
    build_issues.append({"item": "imagery_metadata", "detail": str(exc)})
register_source("esri_imagery", "Esri World Imagery",
    citation = imagery_credit, url = imagery_url,
    licence = "Online imagery service; provider terms and attribution apply.",
    note = "Context imagery dates vary. It is not the dated source for the glacier inventories.")

# Copy local documentation into the export so the credit panel can open it.
metadata_export = asset_dir / "metadata"
metadata_export.mkdir(exist_ok = True)
for source in source_register.values():
    relative = source["metadata"].get("local_metadata")
    if relative and project_path(relative).is_file():
        destination = metadata_export / (source["id"] + project_path(relative).suffix)
        shutil.copy2(project_path(relative), destination)
        source["metadata"]["documentation_download"] = destination.relative_to(explorer_dir).as_posix()

for key in ["synthesis_metadata", "geomorphology_analysis_metadata"]:
    path = synthesis_dir / f"{key}.json"
    if path.is_file():
        provenance_register[key] = {"metadata_file": path.relative_to(project_dir).as_posix(), "content": read_json(path)}
# Keep additional saved processing settings, rather than guessing source scene IDs.
for parent in [map_data_dir, processed_dir / "glacier_outlines_final", processed_dir / "coherence_watershed_v5_1000m_lock", processed_dir / "velocity", processed_dir / "geomorphology" / "field_surveys" / "reconstruction"]:
    if parent.is_dir():
        for path in parent.glob("*.json"):
            if path.stat().st_size < 2_000_000 and path not in metadata_paths.values():
                provenance_register[path.stem] = {"metadata_file": path.relative_to(project_dir).as_posix(), "content": read_json(path)}

technical_references = [
    {"title": "Rasterio: virtual warping and bounded raster reads", "url": "https://rasterio.readthedocs.io/en/stable/topics/virtual-warping.html"},
    {"title": "Rasterio: validity masks and NoData", "url": "https://rasterio.readthedocs.io/en/stable/topics/masks.html"},
    {"title": "Rasterio: rasterisation and polygon masks", "url": "https://rasterio.readthedocs.io/en/stable/topics/features.html"},
    {"title": "Shapely: set difference for lost glacier footprint", "url": "https://shapely.readthedocs.io/en/stable/reference/shapely.difference.html"},
    {"title": "Leaflet 1.9.4: maps, GeoJSON, image overlays and event propagation", "url": "https://leafletjs.com/reference-1.9.4.html"},
    {"title": "Plotly.js: 3D surface plots", "url": "https://plotly.com/javascript/3d-surface-plots/"},
    {"title": "Plotly.js: camera, axes and 3D scene controls", "url": "https://plotly.com/javascript/reference/layout/scene/"},
    {"title": "IPython: IFrame notebook display", "url": "https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.IFrame"},
    {"title": "Tielidze, Mackintosh & Yang (2025): Heard Island glacier change", "url": "https://doi.org/10.5194/tc-19-2677-2025"}]

## Catalogue the available products

The catalogue records paths, source dependencies and file availability. A missing optional product is reported in `build_issues.json`; it is not silently fabricated.

In [ ]:
## Catalogue existing island and glacier layers ##

layer_catalogue = []


def register_layer(layer_id, title, group, value, source_ids,
        glacier_id = None, **details):
    # Record file availability without opening or resampling the dataset.
    path = project_path(value)
    suffix = path.suffix.lower()
    kind = ("raster" if suffix in [".tif", ".tiff"] else
        "vector" if suffix in [".gpkg", ".shp", ".geojson"] else "data")

    layer_catalogue.append({
        "id": layer_id, "title": title, "group": group, "kind": kind,
        "path": path.relative_to(project_dir).as_posix(),
        "available": path.is_file(), "sources": source_ids,
        "glacier_id": glacier_id, "default_visible": False, **details})


# Dated outline layers are selected from the shared inventory GeoJSON.
for year in map_metadata["animation"]["inventory_years"]:
    sources = (["historical_outlines", "inventory_2014"] if year < 2014 else
        ["inventory_2014"] if year == 2014 else
        ["sentinel_1", "inventory_2014", "dem_2002", "coastline", "lagoons"])

    register_layer(f"ice_{year}", f"{year} glacier outlines", "Retreat",
        map_data_dir / "glacier_inventories.geojson", sources,
        year = year,
        feature_filter = {"year": year},
        accepted_names = focus_names if year > 2014 else None,
        other_recent_outlines = "provisional" if year > 2014 else None,
        provenance = "animation")

# Register the surface, thickness, uncertainty and modelled bed outputs.
for key, title, filename, sources in [
        ("surface", "Surface elevation: 2002 DEM",
            "Heard_surface_DEM_2002_50m.tif", ["dem_2002", "coastline"]),
        ("thickness", "Modelled ice thickness",
            "Heard_Millan_thickness_land_50m.tif",
            ["thickness", "coastline", "lagoons"]),
        ("thickness_error", "Reported ice-thickness error",
            "Heard_Millan_thickness_error_land_50m.tif",
            ["thickness", "coastline", "lagoons"]),
        ("bed", "Modelled subglacial elevation",
            "Heard_bed_estimate_candidate_50m.tif",
            ["dem_2002", "thickness", "coastline", "lagoons"])]:

    register_layer(key, title, "Geomorphology",
        processed_dir / "geomorphology" / filename, sources,
        provenance = "bed")

# Surface and bed derivatives retain separate source dependencies.
for prefix, sources in [
        ("", ["dem_2002", "coastline"]),
        ("Heard_bed_", ["dem_2002", "thickness", "coastline", "lagoons"])]:

    for derivative in ["slope", "aspect", "hillshade"]:
        filename = f"{prefix}{derivative}{'_50m' if prefix else ''}.tif"
        label = "Modelled bed" if prefix else "Surface"

        register_layer(f"{label.lower().replace(' ', '_')}_{derivative}",
            f"{label} {derivative}", "Geomorphology",
            processed_dir / "geomorphology" / "terrain" / filename,
            sources)

# Classify the glacier-specific velocity outputs into useful map groups.
terrain_keys = {
    "dem_120m", "elevation_band", "surface_slope_120m",
    "surface_aspect_120m", "surface_aspect_strength_120m"}

quality_keys = {
    "raw_observation_count", "filtered_observation_count",
    "vector_coherence", "quality_mask", "years_represented",
    "median_pair_error", "median_pair_speed_error_ratio"}

for glacier_id, glacier in map_metadata["velocity"]["glaciers"].items():
    for key, value in glacier.get("layers", {}).items():
        group = ("Geomorphology" if key in terrain_keys else
            "Data quality" if key in quality_keys else "Velocity")

        sources = ["inventory_2014"]

        if key not in terrain_keys:
            sources.append("its_live")

        if key in terrain_keys or key == "lower_third_speed":
            sources.append("dem_2002")

        register_layer(f"{glacier_id}_{key}",
            key.replace("_", " ").capitalize(), group, value, sources,
            glacier_id = glacier_id,
            glacier_name = glacier["name"],
            provenance = "velocity")

# Register the reconstructed field layers with their archive source links.
for glacier_id, glacier in map_metadata["field"]["glaciers"].items():
    for key, value in glacier.items():
        if key.endswith("_geojson"):
            title = key.replace("_geojson", "").replace("_", " ").capitalize()

            register_layer(f"{glacier_id}_{key}", title, "Field evidence",
                value, ["field_2363", "campaign_archive"],
                glacier_id = glacier_id,
                provenance = "field")

    for key, value in glacier.get("lagoon_rasters", {}).items():
        register_layer(f"{glacier_id}_lagoon_{key}",
            f"Lagoon: {key.replace('_', ' ')}", "Field evidence",
            value, ["field_2363"],
            glacier_id = glacier_id,
            vertical_reference = (
                "2004 survey water surface = 0; not m a.s.l."),
            provenance = "field")

## Prepare inventories and vector evidence

The coastline is converted to line geometry. Lagoon extents retain their polygon geometry. The overview loss layer is calculated as **1947 footprint minus 2026 footprint**, before any display simplification. Field datasets retain their recorded glacier association.

In [ ]:
## Connect vector evidence and correct the overview geometry ##


def add_map_layer(layer_id, title, group, path, sources, **details):
    layer_catalogue[:] = [r for r in layer_catalogue if r["id"] != layer_id]
    register_layer(layer_id, title, group, path, sources, **details)


focus_sources = ["dem_2002", "thickness", "inventory_2014", "sentinel_1", "coastline", "lagoons", "project_analysis"]
retreat_sources = ["historical_outlines", "inventory_2014", "sentinel_1", "dem_2002", "coastline", "lagoons", "project_analysis"]
for key, title, group, path, sources in [
        ("geology", "Mapped geology", "Geology", project_dir / "Output_Data/geology_clipped.shp", ["geology", "coastline"]),
        ("coastline", "Coastline", "Context", input_dir / "HIMI_Features/himi_coastline_py.gpkg", ["coastline"]),
        ("lagoons", "Lagoons · 2014 extent", "Context", input_dir / "heard_island_shp/UpdatedHI_Lagoons2014.shp", ["lagoons"]),
        ("soundings_2004", "Lagoon soundings · 2004", "Field evidence", map_data_dir / "lagoon_soundings_2004.geojson", ["field_2363"]),
        ("retreat_zones", "Retreat intervals & remaining 2026 ice", "Retreat", synthesis_dir / "retreat_zones.gpkg", retreat_sources),
        ("transects", "Geomorphology transects", "Geomorphology", synthesis_dir / "geomorphology_transects.gpkg", focus_sources),
        ("bed_highs", "Screened modelled bed highs", "Geomorphology", synthesis_dir / "candidate_bed_highs.csv", focus_sources),
        ("cross_sections", "Cross-section sampling lines", "Geomorphology", synthesis_dir / "glacier_cross_sections.csv", focus_sources)]:
    add_map_layer(key, title, group, path, sources)

for glacier in glacier_catalogue.itertuples():
    if glacier.name_1 in focus_names:
        add_map_layer(f"{glacier.RGIId}_summer_incidence", "Potential summer solar exposure", "Surface exposure",
            synthesis_dir / f"{glacier.name_1}_summer_incidence_index.tif", ["dem_2002", "inventory_2014", "project_analysis"], glacier_id = glacier.RGIId)
for key, title in [("median_speed", "Median surface speed · 2016–2022"),
        ("year_balanced_speed", "Year-balanced speed · 2017–2022"), ("vector_coherence", "Flow-direction consistency")]:
    add_map_layer(f"island_{key}", title, "Velocity", processed_dir / "velocity" / f"Heard_{key}.tif",
        ["its_live", "inventory_2014", "project_analysis"], provenance = "velocity")

field_titles = {"bed_sections_geojson": "Interpreted radar sections · 2000",
    "lagoon_contours_geojson": "Lagoon depth contours · 2004", "shore_track_geojson": "Partial shoreline survey · 2003",
    "lagoon_2000_points_geojson": "Lagoon soundings · 2000"}
for record in layer_catalogue:
    for suffix, title in field_titles.items():
        if record["id"].endswith(suffix):
            record["title"] = title
    if record["id"] == "soundings_2004":
        record["glacier_id"] = "RGI60-19.00023"

# Keep metric coordinates for spatial operations; transform only display copies.
map_vectors = {}
coast_polygon = None
for record in layer_catalogue:
    if record["kind"] != "vector" and record["id"] not in ["bed_highs", "cross_sections"]:
        continue
    if record["id"].startswith("ice_") or not record["available"]:
        continue
    path = project_path(record["path"])
    try:
        if path.suffix.lower() == ".csv":
            table = pd.read_csv(path)
            if record["id"] == "cross_sections":
                rows = [{"name_1": name, "section": section,
                    "geometry": LineString(part.sort_values("offset_m")[["E", "N"]].to_numpy())}
                    for (name, section), part in table.groupby(["name_1", "section"]) if len(part) >= 2]
                if not rows:
                    continue
                frame = gpd.GeoDataFrame(rows, crs = "EPSG:32743")
            else:
                frame = gpd.GeoDataFrame(table, geometry = gpd.points_from_xy(table.E, table.N), crs = "EPSG:32743")
        else:
            frame = gpd.read_file(path)
        if frame.crs is None:
            raise ValueError("No CRS recorded; display refused rather than guessing.")
        frame = frame.loc[frame.geometry.notna() & ~frame.geometry.is_empty].to_crs("EPSG:32743").copy()
        frame.geometry = frame.geometry.map(lambda g: make_valid(force_2d(g)))
        if record["id"] == "coastline":
            # A boundary is a line. Fill opacity alone would leave polygon geometry.
            coast_polygon = frame.copy()
            frame.geometry = frame.geometry.map(lambda g: g.boundary if g.geom_type in ["Polygon", "MultiPolygon"] else g)
        map_vectors[record["id"]] = frame
    except Exception as exc:
        build_issues.append({"item": record["id"], "detail": str(exc)})
        record["display_error"] = str(exc)

# Compute the requested lost footprint from the two original inventory geometries.
lost_rows = []
for name in focus_names:
    start = outlines.loc[outlines.name_1.eq(name) & outlines.year.eq(1947)]
    end = outlines.loc[outlines.name_1.eq(name) & outlines.year.eq(2026)]
    if not start.empty and not end.empty:
        original = unary_union(start.geometry)
        recent = unary_union(end.geometry)
        lost = original.difference(recent)
        if not lost.is_empty:
            assert lost.intersection(recent).area < 0.1
            lost_rows.append({"name_1": name, "interval": "1947 minus 2026", "lost_area_km2": lost.area / 1e6, "geometry": lost})
if lost_rows:
    map_vectors["lost_1947_2026"] = gpd.GeoDataFrame(lost_rows, crs = "EPSG:32743")
    add_map_layer("lost_1947_2026", "Ice lost · 1947–2026 · Brown & Stephenson", "Retreat",
        map_data_dir / "glacier_inventories.geojson", retreat_sources,
        derived_geometry = "1947 footprint minus 2026 footprint, calculated in EPSG:32743")

# Use the main Heard Island polygon, not the remote McDonald Islands, for the opening view.
if coast_polygon is not None and not coast_polygon.empty:
    pieces = coast_polygon.explode(index_parts = False).reset_index(drop = True)
    main_coast = pieces.loc[pieces.geometry.area.idxmax()].geometry
    island_domain = main_coast.buffer(700)
else:
    island_domain = unary_union(outlines.geometry).buffer(1200)

rock_colours = {}
if "geology" in map_vectors and "Rock_Type" in map_vectors["geology"]:
    palette = ["#FFB36B", "#69D5FF", "#D6E778", "#C596FF", "#FF7B95", "#80DEAD", "#C8B8A5", "#7CA4E5", "#F4D875", "#F68CF9", "#B6C9D7", "#E09A61"]
    rock_colours = {rock: palette[i % len(palette)] for i, rock in enumerate(sorted(map_vectors["geology"].Rock_Type.fillna("Unknown").astype(str).unique()))}

tooltip_labels = {"name_1": "Glacier", "Rock_Type": "Mapped rock/deposit type", "zone": "Retreat interval", "interval": "Comparison",
    "candidate": "Screened bed high", "section": "Sampling section", "profile": "Radar profile", "survey_year": "Survey year",
    "candidate_bed_m": "Modelled bed elevation (m)", "supported_trials": "Trials with data", "trials_with_nearby_high": "Trials retaining a nearby high",
    "depth_m": "Depth below survey water (m)", "geometry_status": "Registration status", "lost_area_km2": "Lost footprint (km²)"}


def display_geojson(frame, tolerance = 4):
    # Simplification is for the screen only. Areas and set differences use full geometry.
    part = frame.copy()
    part.geometry = part.geometry.simplify(tolerance, preserve_topology = True)
    return json.loads(part.to_crs("EPSG:4326").to_json(drop_id = True, na = "null"))


def geographic_bounds(domain):
    west, south, east, north = transform_bounds("EPSG:32743", "EPSG:4326", *domain.bounds)
    return [[south, west], [north, east]]


records = {r["id"]: r for r in layer_catalogue}
velocity_path = map_data_dir / "Heard_velocity_summary.geojson"
velocity_summary = {f["properties"]["RGIId"]: f["properties"] for f in read_json(velocity_path)["features"]} if velocity_path.is_file() else {}
evidence_path = synthesis_dir / "focus_evidence_summary.csv"
focus_evidence = pd.read_csv(evidence_path, index_col = "name_1") if evidence_path.is_file() else pd.DataFrame()
print(f"Prepared {len(map_vectors)} vector datasets. Lost-footprint polygons calculated from 1947 minus 2026.")

## Define raster legends

Equivalent products use the same limits across glaciers. Aspect uses a circular palette; categorical flags use discrete colours. Scale saturation is stated in the interface. These definitions run before all raster/dashboard preparation.

In [ ]:
## Define comparable raster scales before any dashboard preparation ##

# Explicit scales make comparisons consistent between glaciers and dates.
elevation_palette = [
    "#293F86", "#268C87", "#A0AC73", "#C5A37A", "#F6F1DC"]
speed_palette = [
    "#36196B", "#315EAD", "#27B7B5", "#F6DF72", "#F27945"]
aspect_palette = [
    "#5CC8FF", "#FFD166", "#EF729C", "#A5E075", "#5CC8FF"]
solar_palette = ["#4D88D1", "#E8EFF5", "#F49B51"]
specifications = []


def raster_spec(layer_id, title, group, limits, colours, note, ticks):
    specifications.append({
        "id": layer_id, "title": title, "group": group,
        "limits": limits, "colours": colours,
        "note": note, "ticks": ticks})


for key, title, note in [
        ("surface", "Surface elevation · 2002",
            "Elevation in metres using the source DEM's vertical reference."),
        ("bed", "Modelled subglacial elevation",
            "2002 surface minus modelled thickness. "
            "Mixed observation periods; not a surveyed bed.")]:
    raster_spec(key, title, "Elevation", [-200, 3000],
        elevation_palette, note,
        ["≤ −200 m", "1,400 m", "≥ 3,000 m"])

for key, title, maximum, note in [
        ("thickness", "Modelled ice thickness", 600,
            "Modelled thickness, representative of 2010–2020; "
            "50 m working grid."),
        ("thickness_error", "Reported thickness error", 150,
            "Reported model error in metres; not the total uncertainty "
            "of the derived bed.")]:
    raster_spec(key, title, "Elevation", [0, maximum],
        speed_palette, note,
        ["0 m", f"{maximum / 2:g} m", f"≥ {maximum} m"])

for prefix, label in [
        ("surface", "Surface"), ("modelled_bed", "Modelled bed")]:
    raster_spec(f"{prefix}_slope", f"{label} slope", "Terrain",
        [0, 60], ["#22214D", "#2E94A0", "#F5E68C", "#F17450"],
        "Slope angle in degrees. Bed derivatives inherit the bed model's limitations."
        if prefix == "modelled_bed" else
        "Slope derived from the accepted 2002 DEM.",
        ["0°", "30°", "≥ 60°"])

    raster_spec(f"{prefix}_aspect", f"{label} aspect", "Terrain",
        [0, 360], aspect_palette,
        "Downhill direction, clockwise from north; "
        "aspect has no meaning on flat cells.",
        ["N", "E", "S", "W", "N"])

    raster_spec(f"{prefix}_hillshade", f"{label} hillshade", "Terrain",
        [0, 255], ["#142338", "#F0F4F7"],
        "Artificial illumination for terrain shape; "
        "not measured solar exposure.",
        ["Dark", "", "Light"])

for key, title, period in [
        ("median_speed", "Median surface speed · 2016–2022",
            "Median across accepted image-pair observations."),
        ("year_balanced_speed", "Year-balanced surface speed · 2017–2022",
            "Median of available annual pixel medians.")]:
    raster_spec(f"island_{key}", title, "Velocity", [0, 1000],
        speed_palette,
        period + " 120 m grid; transparent gaps have no accepted velocity value.",
        ["0", "500", "≥ 1,000 m yr⁻¹"])

raster_spec("island_vector_coherence", "Flow-direction consistency",
    "Velocity", [0, 1], ["#5E294A", "#E6C572", "#44C5BA"],
    "Vector coherence: nearer 1 means more consistent observed flow direction; "
    "not a confidence probability.",
    ["0 · mixed", "0.5", "1 · consistent"])

for glacier in glacier_catalogue.itertuples():
    if glacier.name_1 in focus_names:
        raster_spec(f"{glacier.RGIId}_summer_incidence",
            f"{glacier.name_1} · Potential summer solar exposure",
            "Surface exposure", [0.7, 1.3], solar_palette,
            "Sun-incidence index relative to a horizontal surface: "
            "1 = equal. Geometry only, not measured energy or melt.",
            ["≤ 0.70 · lower", "1 · equal", "≥ 1.30 · higher"])

raster_spec("RGI60-19.00023_lagoon_depth",
    "Brown lagoon · Depth below 2004 survey water",
    "Field evidence", [0, 60], ["#DBF1E7", "#4CAAC3", "#21418E"],
    "Supported interpolation of 2004 soundings; depth is relative to "
    "survey water, not sea level. Gaps are unsurveyed.",
    ["0 m", "30 m", "60 m"])


spec_lookup = {s["id"]: s for s in specifications}

# Reuse the overview scales for equivalent glacier-level products.
glacier_specs = {}

for key, overview_key in {
        "median_speed": "island_median_speed",
        "year_balanced_speed": "island_year_balanced_speed",
        "vector_coherence": "island_vector_coherence",
        "dem_120m": "surface",
        "lower_third_speed": "island_median_speed",
        "surface_slope_120m": "surface_slope",
        "surface_aspect_120m": "surface_aspect"}.items():
    glacier_specs[key] = dict(spec_lookup[overview_key])

glacier_specs["dem_120m"]["title"] = "2002 surface · Velocity grid"
glacier_specs["lower_third_speed"].update(
    title = "Lower-elevation third · Median speed",
    note = "Accepted speed within the lowest third of the glacier's "
        "2002 elevation range; not its lowest third by area.")
glacier_specs["surface_slope_120m"]["title"] = "Surface slope · Velocity grid"
glacier_specs["surface_aspect_120m"]["title"] = "Surface aspect · Velocity grid"

for key, title, limits, ticks, note in [
        ("median_vx", "Easting velocity component",
            [-1000, 1000], ["≤ −1,000", "0", "≥ 1,000 m yr⁻¹"],
            "Signed component along the projected easting axis; "
            "not total speed."),
        ("median_vy", "Northing velocity component",
            [-1000, 1000], ["≤ −1,000", "0", "≥ 1,000 m yr⁻¹"],
            "Signed component along the projected northing axis; "
            "not total speed."),
        ("raw_observation_count", "Available image-pair observations",
            [0, 200], ["0", "100", "200 pairs"],
            "Available observations before the final quality selection."),
        ("filtered_observation_count", "Accepted image-pair observations",
            [0, 200], ["0", "100", "200 pairs"],
            "Observations retained by the saved processing filters; "
            "not independent annual samples."),
        ("years_represented", "Years represented · 2017–2022",
            [0, 6], ["0", "3", "6 years"],
            "Number of represented annual estimates in the "
            "year-balanced analysis."),
        ("surface_aspect_strength_120m", "Local aspect consistency",
            [0, 1], ["0 · mixed", "0.5", "1 · consistent"],
            "Length of the mean aspect vector within a velocity-grid "
            "cell; not an uncertainty probability."),
        ("median_pair_error", "Median reported pair-speed error",
            [0, 200], ["0", "100", "≥ 200 m yr⁻¹"],
            "Median reported error of accepted image pairs; not a "
            "confidence interval for the final median speed."),
        ("median_pair_speed_error_ratio", "Pair error relative to speed",
            [0, 1], ["0", "0.5", "≥ 1"],
            "Saved pair-error-to-speed diagnostic; larger values indicate "
            "greater error relative to measured speed.")]:
    glacier_specs[key] = {
        "title": title, "limits": limits,
        "colours": solar_palette
            if key in ["median_vx", "median_vy"] else speed_palette,
        "ticks": ticks, "note": note}

# Categorical products need discrete colours, not a continuous gradient.
for key, title, limits, colours, ticks, note in [
        ("quality_mask", "Velocity quality selection",
            [0, 1], ["#F08091", "#68DAB0"], ["Rejected", "Retained"],
            "Saved selection flag: rejected cells are distinct from NoData."),
        ("elevation_band", "Glacier elevation thirds",
            [1, 3], ["#FFB36B", "#69D5FF", "#D6E778"],
            ["Lower", "Middle", "Upper"],
            "Equal thirds of the glacier's 2002 elevation range; "
            "not equal areas.")]:
    glacier_specs[key] = {
        "title": title, "limits": limits, "colours": colours,
        "ticks": ticks, "note": note, "discrete": True}



# Field support and uncertainty diagnostics remain in Brown's dashboard.
for key, title, limits, colours, ticks, note, discrete in [
        ("relative_bed", "Lagoon bed relative to survey water", [-60, 0], ["#21418E", "#4CAAC3", "#DBF1E7"], ["−60 m", "−30 m", "0 m"], "Negative depth below 2004 survey water; not elevation above sea level.", False),
        ("nearest_sounding", "Distance to nearest sounding", [0, 100], speed_palette, ["0 m", "50 m", "≥ 100 m"], "Distance to the nearest observed sounding; a support diagnostic, not depth uncertainty.", False),
        ("triangle_max_edge", "Largest interpolation-triangle edge", [0, 150], speed_palette, ["0 m", "75 m", "≥ 150 m"], "Spatial support diagnostic used to exclude long interpolation triangles.", False),
        ("support", "Lagoon interpolation support", [0, 1], ["#F08091", "#68DAB0"], ["Outside support", "Supported"], "The saved support mask. Unsupported is distinct from a zero depth.", True)]:
    key = f"RGI60-19.00023_lagoon_{key}"
    if key in records:
        specifications.append({"id": key, "title": title, "group": "Field evidence", "limits": limits,
            "colours": colours, "ticks": ticks, "note": note, "discrete": discrete})
spec_lookup = {s["id"]: s for s in specifications}
print(f"Raster styling is defined: {len(specifications)} shared products and {len(glacier_specs)} glacier velocity/quality products.")

## Create bounded browser images

Rasterio reads a limited display window through `WarpedVRT`. Nearest-neighbour sampling preserves categorical flags and avoids arithmetic averaging of circular aspect values. Source masks and polygon masks both apply; valid zero values remain valid. PNGs are cached by source stamp, viewing domain and style.

Technical references: [Rasterio virtual warping](https://rasterio.readthedocs.io/en/stable/topics/virtual-warping.html), [validity masks](https://rasterio.readthedocs.io/en/stable/topics/masks.html), [raster features](https://rasterio.readthedocs.io/en/stable/topics/features.html).

In [ ]:
## Prepare small display images without reading full rasters into memory ##


def bounded_raster(record, spec, domain, tag, maximum_side):
    """Warp a bounded display clip, honouring both NoData and validity masks."""
    path = project_path(record["path"])
    if not path.is_file():
        return None
    stamps = [[p.name, p.stat().st_size, p.stat().st_mtime_ns]
        for p in [path, Path(str(path) + ".msk")] if p.is_file()]
    signature = {"version": 3, "path": record["path"], "stamps": stamps,
        "domain": domain.wkb_hex, "style": spec, "maximum_side": maximum_side}
    digest = hashlib.sha256(json.dumps(signature, sort_keys = True).encode()).hexdigest()[:16]
    png_path = raster_cache_dir / f"{tag}_{record['id']}_{digest}.png"
    info_path = png_path.with_suffix(".json")
    if png_path.is_file() and info_path.is_file():
        return {**read_json(info_path), "image": png_path.relative_to(explorer_dir).as_posix()}
    web_domain = shape(transform_geom("EPSG:32743", "EPSG:3857", mapping(domain)))
    with rasterio.Env(GDAL_CACHEMAX = 64 * 1024 * 1024):
        with rasterio.open(path) as src:
            if src.crs is None:
                raise ValueError(f"Raster has no CRS: {path}")
            native, width, height = calculate_default_transform(src.crs, "EPSG:3857", src.width, src.height, *src.bounds)
            sb = transform_bounds(src.crs, "EPSG:3857", *src.bounds)
            db = web_domain.bounds
            west, south, east, north = max(sb[0], db[0]), max(sb[1], db[1]), min(sb[2], db[2]), min(sb[3], db[3])
            if west >= east or south >= north:
                return None
            width = max(1, math.ceil((east - west) / abs(native.a)))
            height = max(1, math.ceil((north - south) / abs(native.e)))
            reduction = min(1, maximum_side / max(width, height))
            width, height = max(1, math.ceil(width * reduction)), max(1, math.ceil(height * reduction))
            transform = from_bounds(west, south, east, north, width, height)
            with WarpedVRT(src, crs = "EPSG:3857", transform = transform, width = width, height = height,
                    dtype = "float32", nodata = np.nan, resampling = Resampling.nearest) as vrt:
                values = vrt.read(1, masked = True)
            native_grid = {"crs": src.crs.to_string(), "shape": [src.height, src.width],
                "pixel_size": [float(v) for v in src.res], "units": src.units[0],
                "nodata": str(src.nodata), "tags": src.tags()}
    inside = geometry_mask([mapping(web_domain)], values.shape, transform, invert = True, all_touched = False)
    valid = inside & ~np.ma.getmaskarray(values) & np.isfinite(values.data)
    if not valid.any():
        return None
    low, high = spec["limits"]
    if spec.get("discrete"):
        cmap = ListedColormap(spec["colours"])
        normalise = BoundaryNorm(np.arange(low - 0.5, high + 1), cmap.N, clip = True)
    else:
        cmap = LinearSegmentedColormap.from_list("explorer", spec["colours"], N = 256)
        normalise = Normalize(low, high, clip = True)
    rgba = cmap(normalise(np.where(valid, values.data, low)), bytes = True)
    rgba[..., 3] = np.where(valid, 255, 0).astype("uint8")
    Image.fromarray(rgba).save(png_path, optimize = True)
    west, south, east, north = transform_bounds("EPSG:3857", "EPSG:4326", west, south, east, north)
    info = {"bounds": [[south, west], [north, east]], "display_shape": [height, width],
        "native_grid": native_grid, "display_crs": "EPSG:3857", "resampling": "nearest",
        "source_signature": digest}
    write_json(info_path, info)
    record["native_grid"] = native_grid
    return {**info, "image": png_path.relative_to(explorer_dir).as_posix()}


def raster_layer(record, spec, domain, tag, maximum_side, tab):
    try:
        prepared = bounded_raster(record, spec, domain, tag, maximum_side)
    except Exception as exc:
        build_issues.append({"item": f"{tag}/{record['id']}", "detail": str(exc)})
        return None
    if prepared is None:
        return None
    record["native_grid"] = prepared["native_grid"]
    return {"id": record["id"], "title": spec["title"], "tab": tab, "type": "raster",
        "image": prepared["image"], "bounds": prepared["bounds"], "style": spec,
        "sources": record["sources"], "path": record["path"], "native_grid": prepared["native_grid"]}


def vector_layer(key, frame, domain = None, glacier_id = None, name = None):
    record = records[key]
    if record.get("glacier_id") not in [None, glacier_id]:
        return None
    part = frame.copy()
    if glacier_id:
        if "name_1" in part:
            part = part.loc[part.name_1.isna() | part.name_1.eq(name)]
        if "RGIId" in part:
            part = part.loc[part.RGIId.isna() | part.RGIId.eq(glacier_id)]
    if domain is not None:
        part = part.loc[part.geometry.intersects(domain)].copy()
        if part.empty:
            return None
        part.geometry = part.geometry.intersection(domain)
    part = part.loc[~part.geometry.is_empty]
    if part.empty:
        return None
    fields = [field for field in tooltip_labels if field in part]
    return {"id": key, "title": record["title"], "tab": {"Context": "Overview"}.get(record["group"], record["group"]),
        "type": "vector", "data": display_geojson(part[[*fields, "geometry"]]),
        "sources": record["sources"], "path": record["path"]}


def inventory_history(glacier_id = None):
    frame = outlines if glacier_id is None else outlines.loc[outlines.RGIId.eq(glacier_id)]
    result = display_geojson(frame)
    for feature in result["features"]:
        p = feature["properties"]
        p["provisional"] = p["year"] > 2014 and p["name_1"] not in focus_names
        p["status"] = "Provisional recent outline" if p["provisional"] else "Project inventory"
        p["sources"] = records[f"ice_{p['year']}"]["sources"]
    return result["features"]


def value_text(value, suffix = "", digits = 1):
    return "Unavailable" if value is None or pd.isna(value) else f"{float(value):,.{digits}f}{suffix}"


def metrics_table(values, fields):
    return "<table>" + "".join(f"<tr><th>{escape(label)}</th><td>{value_text(values.get(key), unit, digits)}</td></tr>"
        for key, label, unit, digits in fields) + "</table>"

## Prepare the island overview

Include all available 2019 and 2026 inventories, including provisional context outside the focus pair. Retain unmatched historical ice in the island view. Field evidence, cross-section sampling lines, bed rasters and flow-direction consistency are excluded from this menu.

In [ ]:
## Build a focused island overview ##

# Explicit scope rules keep field details and bed products in glacier dashboards.
excluded_overview = {"retreat_zones", "cross_sections", "bed", "modelled_bed_slope",
    "modelled_bed_aspect", "modelled_bed_hillshade", "island_vector_coherence"}
overview_layers = []
for key, frame in map_vectors.items():
    record = records[key]
    if key in excluded_overview or record["group"] == "Field evidence" or record.get("glacier_id"):
        continue
    layer = vector_layer(key, frame, island_domain)
    if layer:
        overview_layers.append(layer)
for spec in specifications:
    record = records.get(spec["id"])
    if record is None or spec["id"] in excluded_overview or record["group"] == "Field evidence":
        continue
    layer = raster_layer(record, spec, island_domain, "island", MAX_ISLAND_IMAGE_SIDE,
        {"Elevation": "Geomorphology", "Terrain": "Geomorphology"}.get(spec["group"], spec["group"]))
    if layer:
        overview_layers.append(layer)

# Preserve island-wide historical records that could not be matched to a named glacier.
island_history = inventory_history()
island_inventory_path = map_data_dir / "island_inventories.geojson"
unmatched_history = []
if island_inventory_path.is_file():
    whole = gpd.read_file(island_inventory_path).to_crs("EPSG:32743")
    for year in years:
        part = whole.loc[whole.year.eq(year)]
        named = outlines.loc[outlines.year.eq(year)]
        if part.empty:
            continue
        unmatched = unary_union(part.geometry).difference(unary_union(named.geometry))
        if not unmatched.is_empty and unmatched.area > 1:
            frame = gpd.GeoDataFrame([{"year": year, "name_1": "Unassigned inventory ice", "provisional": year > 2014,
                "status": "Unmatched inventory geometry", "geometry": unmatched}], crs = "EPSG:32743")
            unmatched_history.extend(display_geojson(frame)["features"])
island_history += unmatched_history

# Missing dates are reported instead of substituting a different glacier's outline.
inventory_counts = {str(year): int(outlines.loc[outlines.year.eq(year), "RGIId"].nunique()) for year in years}
selector = []
for glacier in glacier_catalogue.itertuples():
    part = outlines.loc[outlines.RGIId.eq(glacier.RGIId)]
    label_geometry = part.loc[part.year.eq(2014)]
    if label_geometry.empty:
        label_geometry = part.loc[part.year.eq(part.year.max())]
    label_point = unary_union(label_geometry.geometry).representative_point()
    label_lon, label_lat = transform_geom("EPSG:32743", "EPSG:4326", mapping(label_point))["coordinates"]
    selector.append({"id": glacier.RGIId, "name": glacier.name_1, "focus": glacier.name_1 in focus_names,
        "label_anchor": [label_lat, label_lon],
        "bounds": geographic_bounds(unary_union(part.geometry).buffer(DASHBOARD_BUFFER_M)),
        "years": sorted(int(y) for y in part.year.unique()),
        "data": f"assets/data/{glacier.RGIId}.js", "terrain": f"assets/data/{glacier.RGIId}_terrain.js"})
print("Glaciers represented by inventory year:", inventory_counts)
print(f"Island overview: {len(overview_layers)} thematic layers, plus the five inventory dates.")

## Prepare glacier dashboard data

Each glacier gets its own data asset. The browser opens it on demand. Maps include dated ice, geology, geomorphology, velocity and quality tabs; Brown and Stephenson receive the additional products available for them. Reported statistics come from existing summaries, not from downsampled display images.

In [ ]:
## Prepare glacier dashboard data, one glacier at a time ##

# All style definitions are executed above. No Folium HTML representation interrupts the build.
quality_metrics = {"vector_coherence", "raw_observation_count", "filtered_observation_count", "quality_mask",
    "years_represented", "median_pair_error", "median_pair_speed_error_ratio"}
terrain_metrics = {"dem_120m", "elevation_band", "surface_slope_120m", "surface_aspect_120m", "surface_aspect_strength_120m"}
dashboard_manifest = []

for glacier in glacier_catalogue.itertuples():
    glacier_id, name = glacier.RGIId, glacier.name_1
    history = inventory_history(glacier_id)
    domain = unary_union(outlines.loc[outlines.RGIId.eq(glacier_id)].geometry).buffer(DASHBOARD_BUFFER_M)
    layers = []
    for key, frame in map_vectors.items():
        layer = vector_layer(key, frame, domain, glacier_id, name)
        if layer:
            layers.append(layer)
    choices = [(records[s["id"]], s,
        {"Elevation": "Geomorphology", "Terrain": "Geomorphology", "Surface exposure": "Solar"}.get(s["group"], s["group"]))
        for s in specifications if s["id"] in records and not s["id"].startswith("island_")
        and records[s["id"]].get("glacier_id") in [None, glacier_id]]
    for key, spec in glacier_specs.items():
        record = records.get(f"{glacier_id}_{key}")
        if record:
            choices.append((record, spec, "Quality" if key in quality_metrics else "Geomorphology" if key in terrain_metrics else "Velocity"))
    for record, spec, tab in choices:
        layer = raster_layer(record, spec, domain, glacier_id, MAX_GLACIER_IMAGE_SIDE, tab)
        if layer:
            layers.append(layer)

    areas = "".join(f"<tr><th>{f['properties']['year']}{' *' if f['properties']['provisional'] else ''}</th>"
        f"<td>{f['properties']['area_km2']:.2f} km²</td></tr>" for f in sorted(history, key = lambda f: f['properties']['year']))
    flow = velocity_summary.get(glacier_id, {})
    missing_years = [str(y) for y in years if y not in [f["properties"]["year"] for f in history]]
    recent_note = ("Recent outlines for this focus glacier are the project analysis versions." if name in focus_names else
        "* 2019 and 2026 outlines are provisional context. They are displayed for comparison, not promoted to accepted analysis results.")
    notes = {
        "Overview": "<p>Choose a tab to explore the evidence. Layers stay on when you change tabs; the active-layer count helps track what is visible.</p>"
            "<p>The viewing area includes all available dated footprints and 600 m of surrounding terrain.</p>",
        "Retreat": "<h3>Mapped glacier area</h3><table>" + areas + "</table><p>" + recent_note + "</p>"
            + ("<p>No saved outline for: " + ", ".join(missing_years) + ". No replacement is invented.</p>" if missing_years else "")
            + "<p>1947 is a nominal inventory date and may include earlier records. Recent no-advance and fixed upper-ice constraints are imposed assumptions. Animation steps through mapped dates; equal screen time does not mean equal elapsed time.</p>",
        "Geology": "<p>Mapped geology describes exposed surface units. A Snow/Ice polygon does not identify the material beneath the glacier. Moraine deposits do not by themselves establish the lithology of an overdeepened bed.</p>",
        "Geomorphology": "<p>Surface terrain uses the accepted 2002 RADARSAT DEM. Modelled subglacial terrain subtracts modelled ice thickness, representative of 2010–2020. It combines observation periods and is not a direct bed survey.</p>"
            "<p>Screened bed highs are hypotheses for local controls. Persistence-test counts are not probabilities or proof of pinning.</p>",
        "Velocity": "<h3>Surface flow · 2016–2022</h3>" + metrics_table(flow, [
            ("median_speed_m_yr", "Median speed", " m yr⁻¹", 1), ("coverage_percent", "Usable coverage", "%", 1),
            ("lower_third_speed_m_yr", "Lower-elevation third", " m yr⁻¹", 1),
            ("lower_third_coverage_percent", "Lower-third coverage", "%", 1),
            ("year_balanced_speed_m_yr", "Year-balanced speed", " m yr⁻¹", 1)])
            + "<p>Velocity uses the fixed 2014 footprint, including areas that later lost ice. Lower thirds divide elevation range, not area. The year-balanced comparison uses 2017–2022.</p>"
            + ("<p>Jacka was excluded from the velocity analysis.</p>" if name == "Jacka" else ""),
        "Quality": "<p>Usable coverage is the proportion of initially observed positive-vector cells retained by the saved filters, not a percentage of the entire glacier polygon. Pair errors are not confidence intervals for the final median. Transparent NoData and rejected cells are different.</p>",
        "Solar": "<p>Potential summer sun incidence is normalised to a horizontal surface: 1 means equal exposure. This terrain-and-sun geometry index is not measured radiation or melt. It inherits the 2002 DEM's slope and aspect.</p>",
        "Field evidence": "<p>Interpreted radar archive sections are local profiles, not a continuous surveyed bed. Provisional registration remains marked. Lagoon depths refer to the 2004 survey water surface; they are not elevations above sea level. Unsupported areas remain transparent.</p>"}
    if name in focus_evidence.index:
        for tab, fields in {
                "Retreat": [("area_loss_1947_2026_percent", "Area loss · 1947–2026", "%", 1),
                    ("recent_transect_retreat_m_yr", "Transect retreat · 2019–2026", " m yr⁻¹", 2)],
                "Geomorphology": [("modelled_thickness_median_m", "Median modelled thickness", " m", 1),
                    ("mean_reported_thickness_error_m", "Mean reported thickness error", " m", 1)],
                "Solar": [("summer_incidence_2014_footprint", "Median incidence index · 2014 footprint", "", 3)]}.items():
            notes[tab] += metrics_table(focus_evidence.loc[name], fields)
    tabs = [tab for tab in notes if tab in ["Overview", "Retreat", "Geology", "Geomorphology", "Velocity", "Quality"]
        or any(layer["tab"] == tab for layer in layers)]
    payload = {"id": glacier_id, "name": name, "bounds": geographic_bounds(domain), "history": history,
        "layers": layers, "tabs": tabs, "notes": notes, "flow": flow,
        "recent_status": recent_note, "terrain": f"assets/data/{glacier_id}_terrain.js"}
    path = asset_dir / "data" / f"{glacier_id}.js"
    write_data_script(path, f'window.HI.glaciers["{glacier_id}"]', payload)
    dashboard_manifest.append({"id": glacier_id, "name": name, "layers": len(layers),
        "years": sorted(f["properties"]["year"] for f in history), "data": path.relative_to(explorer_dir).as_posix()})
    print(f"{name}: {len(layers)} layers; saved dashboard data")
    del payload, layers
    gc.collect()

write_json(explorer_dir / "glacier_dashboard_metadata.json", {
    "display_domain": "Union of all available dated footprints, buffered in EPSG:32743",
    "buffer_m": DASHBOARD_BUFFER_M, "maximum_image_side": MAX_GLACIER_IMAGE_SIDE,
    "recent_outline_status": "Brown and Stephenson analysis versions; other recent outlines remain provisional",
    "statistics": "Existing analysis summaries, not recalculated from display images", "glaciers": dashboard_manifest})

## Prepare 3D surface views

3D uses metric coordinates in EPSG:32743 and a bounded grid. The viewer shows the accepted 2002 surface. Dated footprints animate on fixed terrain; no historical thickness is reconstructed. The modelled bed remains in the analytical products and data payload, ready for the separate glacier ice-object work.

Technical references: [Plotly surface plots](https://plotly.com/javascript/3d-surface-plots/), [camera and scene controls](https://plotly.com/javascript/reference/layout/scene/).


In [ ]:
## Build manoeuvrable 3D terrain with explicit data gaps ##


def read_terrain_grid(record, transform, width, height, domain_mask):
    path = project_path(record["path"])
    if not path.is_file():
        return np.full((height, width), np.nan, dtype = "float32")
    with rasterio.Env(GDAL_CACHEMAX = 64 * 1024 * 1024):
        with rasterio.open(path) as src:
            if src.crs is None:
                raise ValueError(f"No CRS: {path}")
            with WarpedVRT(src, crs = "EPSG:32743", transform = transform,
                    width = width, height = height, dtype = "float32", nodata = np.nan,
                    resampling = Resampling.nearest) as vrt:
                values = vrt.read(1, masked = True)
    valid = domain_mask & ~np.ma.getmaskarray(values) & np.isfinite(values.data)
    return np.where(valid, values.data, np.nan).astype("float32")


def matrix_json(values, digits = 2):
    rounded = np.round(values.astype("float64"), digits)
    return [[float(v) if np.isfinite(v) else None for v in row] for row in rounded]


def terrain_lines(geometry, transform, surface, bed):
    # Drape outlines on each terrain; missing heights break the line rather than invent a bed.
    lines = []
    boundary = geometry.boundary if geometry.geom_type in ["Polygon", "MultiPolygon"] else geometry
    parts = list(boundary.geoms) if hasattr(boundary, "geoms") else [boundary]
    for line in parts:
        if not hasattr(line, "coords"):
            continue
        count = max(2, min(800, math.ceil(line.length / max(abs(transform.a), abs(transform.e))) + 1))
        pts = [line.interpolate(f, normalized = True) for f in np.linspace(0, 1, count)]
        xy = np.array([[p.x, p.y] for p in pts])
        col = np.floor((xy[:, 0] - transform.c) / transform.a).astype(int)
        row = np.floor((xy[:, 1] - transform.f) / transform.e).astype(int)
        inside = (col >= 0) & (col < surface.shape[1]) & (row >= 0) & (row < surface.shape[0])
        zs, zb = np.full(len(pts), np.nan), np.full(len(pts), np.nan)
        zs[inside] = surface[row[inside], col[inside]]
        zb[inside] = bed[row[inside], col[inside]]
        lines.append({"x": np.round((xy[:, 0] - transform.c) / 1000, 4).tolist(),
            "y": np.round((xy[:, 1] - transform.f) / 1000, 4).tolist(),
            "surface": [round(float(z), 2) if np.isfinite(z) else None for z in zs],
            "bed": [round(float(z), 2) if np.isfinite(z) else None for z in zb]})
    return lines


def make_terrain(tag, name, domain, maximum_side, glacier_id = None):
    west, south, east, north = domain.bounds
    cell = max(east - west, north - south) / maximum_side
    width, height = max(2, math.ceil((east - west) / cell)), max(2, math.ceil((north - south) / cell))
    transform = from_bounds(west, south, east, north, width, height)
    inside = geometry_mask([mapping(domain)], (height, width), transform, invert = True)
    surface = read_terrain_grid(records["surface"], transform, width, height, inside)
    if not np.isfinite(surface).any():
        write_data_script(asset_dir / "data" / f"{tag}_terrain.js", f'window.HI.terrain["{tag}"]',
            {"available": False, "name": name, "reason": "The accepted surface DEM was not available for this view."})
        return {"id": tag, "available": False}
    bed = read_terrain_grid(records["bed"], transform, width, height, inside)
    footprint = outlines.loc[outlines.year.isin([2014, 2019])]
    ice_domain = geometry_mask([mapping(unary_union(footprint.geometry))], (height, width), transform, invert = True)
    # Land outside the ice domain stays as surface context. Unknown bed under ice stays unknown.
    terrain_bed = np.where(np.isfinite(bed), bed, np.where(~ice_domain, surface, np.nan))
    bed_available = bool(np.isfinite(bed).any())
    history = outlines if glacier_id is None else outlines.loc[outlines.RGIId.eq(glacier_id)]
    masks, lines = {}, {}
    for year in years:
        part = history.loc[history.year.eq(year)]
        # For the island scene include unmatched historical ice as well.
        geometries = list(part.geometry)
        if glacier_id is None:
            geometries += [shape(transform_geom("EPSG:4326", "EPSG:32743", f["geometry"])) for f in unmatched_history if f["properties"]["year"] == year]
        if not geometries:
            continue
        geometry = unary_union(geometries).intersection(domain)
        mask = geometry_mask([mapping(geometry)], (height, width), transform, invert = True)
        masks[str(year)] = ["".join("1" if cell else "0" for cell in row) for row in mask]
        lines[str(year)] = terrain_lines(geometry, transform, surface, terrain_bed)
    options = []
    for key in ["thickness", "surface_slope", "island_median_speed" if glacier_id is None else f"{glacier_id}_median_speed",
            f"{glacier_id}_summer_incidence" if glacier_id else None]:
        if key not in records:
            continue
        spec = spec_lookup.get(key)
        if spec is None and key.endswith("_median_speed"):
            spec = glacier_specs["median_speed"]
        if spec is None:
            continue
        values = read_terrain_grid(records[key], transform, width, height, inside)
        if np.isfinite(values).any():
            options.append({"id": key, "title": spec["title"], "values": matrix_json(values), "style": spec, "sources": records[key]["sources"]})
    if "geology" in map_vectors:
        geo = map_vectors["geology"]
        if "Rock_Type" in geo:
            geo = geo.loc[geo.geometry.intersects(domain)]
            codes = {rock: i + 1 for i, rock in enumerate(rock_colours)}
            shapes = [(g, codes[str(rock)] if pd.notna(rock) else codes.get("Unknown", 0)) for g, rock in zip(geo.geometry, geo.Rock_Type)]
            if shapes:
                values = rasterize(shapes, out_shape = (height, width), transform = transform, fill = 0).astype("float32")
                values[(values == 0) | ~inside] = np.nan
                options.append({"id": "geology", "title": "Mapped surface geology", "values": matrix_json(values, 0),
                    "style": {"limits": [0.5, len(codes) + 0.5], "colours": list(rock_colours.values()), "discrete": True,
                        "ticks": list(rock_colours), "note": "Surface units only. Snow/Ice does not identify buried lithology."}, "sources": ["geology"]})
    payload = {"available": True, "id": tag, "name": name, "bed_available": bed_available,
        "x": np.round(((np.arange(width) + 0.5) * transform.a) / 1000, 5).tolist(),
        "y": np.round(((np.arange(height) + 0.5) * transform.e) / 1000, 5).tolist(),
        "surface": matrix_json(surface), "bed": matrix_json(terrain_bed),
        "ice": masks, "lines": lines, "options": options,
        "origin_m": [west, north], "shape": [height, width], "display_cell_m": [abs(transform.a), abs(transform.e)],
        "crs": "EPSG:32743", "sources": ["dem_2002", "thickness", "inventory_2014", "historical_outlines", "sentinel_1"],
        "note": "Fixed terrain with dated ice footprints. This is not reconstructed historical surface elevation. The bed combines a 2002 surface and modelled thickness from a different period. Missing bed stays blank under the mapped ice domain; surrounding land is surface context."}
    write_data_script(asset_dir / "data" / f"{tag}_terrain.js", f'window.HI.terrain["{tag}"]', payload)
    return {"id": tag, "available": True, "bed_available": bed_available, "shape": [height, width], "display_cell_m": payload["display_cell_m"]}


terrain_manifest = []
for tag, name, domain, size, glacier_id in [("island", "Heard Island", island_domain, MAX_ISLAND_3D_SIDE, None)] + [
        (g.RGIId, g.name_1, unary_union(outlines.loc[outlines.RGIId.eq(g.RGIId)].geometry).buffer(DASHBOARD_BUFFER_M), MAX_GLACIER_3D_SIDE, g.RGIId)
        for g in glacier_catalogue.itertuples()]:
    try:
        terrain_manifest.append(make_terrain(tag, name, domain, size, glacier_id))
        print(f"3D terrain: {name}")
    except Exception as exc:
        build_issues.append({"item": f"3D {name}", "detail": str(exc)})
        write_data_script(asset_dir / "data" / f"{tag}_terrain.js", f'window.HI.terrain["{tag}"]',
            {"available": False, "name": name, "reason": str(exc)})
    gc.collect()
write_json(explorer_dir / "terrain_display_metadata.json", {"scenes": terrain_manifest,
    "sampling": "Nearest-neighbour display grid in EPSG:32743; original analytical rasters unchanged",
    "retreat": "Mapped inventory dates on fixed terrain; no historical ice-thickness reconstruction",
    "bed_context": "Modelled bed where valid; surface land outside the 2014/2019 ice domain; unknown bed within ice remains NoData"})

## Define the browser interface

The browser interface fills its window, keeps layer lists scrollable, and labels every glacier with its available name or identifier. Floating animation controls collapse independently on the island and glacier maps; collapsing pauses playback and retains the selected footprint. Glacier dashboards dim satellite imagery so evidence layers are easier to see.

Fullscreen uses the browser API where permitted and opens the exported page in a full window when an embedded notebook denies access. Technical reference: [MDN Fullscreen API](https://developer.mozilla.org/en-US/docs/Web/API/Element/requestFullscreen).


In [ ]:
## Page structure and accessible controls ##

# Define the static page and controls used by the exported map.
APP_HTML = r'''

<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Heard Island · Ice, terrain & change</title>
<link rel="stylesheet" href="@@LEAFLET_CSS@@"><link rel="stylesheet" href="assets/explorer.css?v=@@BUILD@@">
</head>
<body>
<div id="app">
<header class="masthead"><div class="brand"><span class="eyebrow">KGG375 · HEARD ISLAND</span><h1>Ice, terrain <span>& change</span></h1></div><div class="header-actions"><span class="date-badge">1947 — 2026</span><button id="island-3d" class="accent">Explore in 3D</button><button id="credits-open">Sources & credits</button><button id="fullscreen" title="Enter or exit full screen" aria-label="Enter or exit full screen"><svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" aria-hidden="true"><path d="M8 3H3v5m13-5h5v5M3 16v5h5m8 0h5v-5"/></svg></button></div></header>
<main class="workspace"><section class="map-shell"><div id="island-map" class="map-canvas" aria-label="Interactive map of Heard Island"></div><div class="map-label"><span class="eyebrow">SOUTHERN INDIAN OCEAN</span><strong>Heard Island</strong><span>Click a glacier to explore its evidence</span></div><div id="island-timeline" class="timeline"></div><div id="map-message" class="map-message" hidden></div></section><aside id="island-sidebar" class="sidebar"><section class="selector-panel"><label for="glacier-select">Find a glacier</label><div class="select-row"><select id="glacier-select"><option value="">Select a glacier…</option></select><button id="glacier-open" class="accent">Open</button></div><div class="focus-row"><button data-glacier="RGI60-19.00023" class="brown">Brown ↗</button><button data-glacier="RGI60-19.00005" class="stephenson">Stephenson ↗</button><button id="island-reset">Reset view</button></div></section><section class="sidebar-heading"><h2>Island layers</h2><button id="island-clear" class="text-button">Clear layers</button></section><div class="basemap-row"><label><input type="checkbox" id="island-imagery" checked> Satellite context</label><span class="muted">Imagery dates vary</span></div><p class="scope-note">2019 & 2026: all available glacier outlines. Recent non-focus outlines are provisional and dotted. 1947 is a nominal date and includes some earlier observations. Field evidence and bed rasters are in glacier dashboards.</p><div id="island-layers"></div><section class="sidebar-foot"><p>Transparent gaps have no supported value. Raster colours can saturate at the ends of their scales.</p><button class="text-button" id="methods-open">Data & display methods ↗</button></section></aside></main>
</div>
<dialog id="glacier-dialog" class="large-dialog"><div class="dialog-shell"><header class="dialog-head"><div><span class="eyebrow">GLACIER DASHBOARD</span><h2 id="glacier-title">Glacier</h2></div><div class="dialog-actions"><button id="glacier-3d" class="accent">Explore in 3D</button><button id="glacier-sources">Sources</button><button data-close="glacier-dialog" class="close" aria-label="Close glacier dashboard">×</button></div></header><nav id="glacier-tabs" class="tabs" aria-label="Glacier evidence sections"></nav><div class="dashboard-grid"><div class="map-shell"><div id="glacier-map" class="map-canvas"></div><div id="glacier-timeline" class="timeline"></div></div><aside id="glacier-sidebar" class="sidebar"><div class="sidebar-heading"><span id="active-layer-count"></span><button id="glacier-clear" class="text-button">Clear layers</button></div><div id="glacier-notes"></div><div id="glacier-layers"></div></aside></div></div></dialog>
<dialog id="terrain-dialog" class="large-dialog terrain-dialog"><div class="dialog-shell"><header class="dialog-head"><div><span class="eyebrow">3D TERRAIN</span><h2 id="terrain-title">Terrain</h2></div><div class="dialog-actions"><button id="terrain-reset">Reset camera</button><button id="terrain-sources">Sources</button><button data-close="terrain-dialog" class="close" aria-label="Close 3D terrain">×</button></div></header><div class="terrain-grid"><div id="terrain-plot" aria-label="Manoeuvrable 3D terrain"></div><aside class="sidebar terrain-controls"><p class="instruction">Drag to orbit · scroll to zoom · right-drag to pan</p><p class="scope-note">Surface terrain · 2002 DEM</p><label for="terrain-colour">Colour the terrain by</label><select id="terrain-colour"><option value="elevation">Elevation</option></select><div id="terrain-legend"></div><label for="terrain-exaggeration">Vertical exaggeration <output id="terrain-exaggeration-label">1×</output></label><input type="range" id="terrain-exaggeration" min="1" max="5" step="0.5" value="1"><label class="check-row"><input type="checkbox" id="terrain-ice" checked> Show dated ice footprint</label><section id="terrain-timeline" class="terrain-timeline"></section><p id="terrain-date-status" class="scope-note"></p><div class="terrain-note"><strong>What this animation shows</strong><p>Mapped ice footprints on fixed terrain. It does not reconstruct historical surface heights or thickness.</p></div><small id="terrain-grid-note"></small></aside></div></div></dialog>
<dialog id="credits-dialog" class="credits-dialog"><div class="dialog-shell"><header class="dialog-head"><div><span class="eyebrow">PROVENANCE</span><h2>Sources, metadata & credits</h2></div><button data-close="credits-dialog" class="close" aria-label="Close sources">×</button></header><div class="credits-content"><p>Each displayed dataset links to its source record. Supplied files with unresolved bibliographic details are identified explicitly; no author, release date or licence is invented.</p><div class="credits-tools"><input id="credits-search" type="search" placeholder="Search sources or layers…" aria-label="Search sources and layers"><button id="credits-all">Show all sources</button><a href="source_register.json" download>Source register ↓</a><a href="layer_catalogue.json" download>Layer metadata ↓</a><a href="processing_provenance.json" download>Processing record ↓</a></div><div id="credits-list"></div><details id="methods-details"><summary>Technical references & display methods</summary><div id="technical-references"></div><p>Display rasters use bounded nearest-neighbour reprojection with source masks. Glacier clips use the union of available dated footprints plus a 600 m buffer. Screen geometries are simplified by 4 m after all area calculations. 3D heights use a bounded grid in EPSG:32743, with distances in kilometres and elevations in metres.</p><p>Recent non-focus outlines remain provisional. Missing inventory dates are not filled from neighbouring glaciers. Retreat animation gives equal screen time to mapped dates, which are unevenly spaced in real time.</p><p>Software: Leaflet 1.9.4 (BSD-2-Clause), Plotly.js 3.1.0 (MIT), and the Python packages recorded in the build manifest.</p></details><details><summary>Layer inventory and availability</summary><div id="credits-layers"></div></details></div></div></dialog>
<div id="toast" role="status" aria-live="polite" hidden></div>
<script>window.HI = {glaciers: {}, terrain: {}};</script>
<script src="@@LEAFLET_JS@@"></script>
<script src="assets/data/overview.js?v=@@BUILD@@"></script>
<script src="assets/explorer.js?v=@@BUILD@@"></script>
<noscript>This interactive map requires JavaScript.</noscript>
</body></html>

'''


In [ ]:
## Responsive layout, full-height map and scrolling panels ##

# Style collapsible panels, readable labels and modal backdrops.
APP_CSS = r'''

:root{color-scheme:dark;--bg:#0b1420;--panel:#111e2c;--panel2:#192a3b;--line:#304355;--text:#e8eff5;--muted:#b3c5d3;--cyan:#69d5ff;--orange:#ffb36b}
*{box-sizing:border-box}html,body{width:100%;height:100%;margin:0;overflow:hidden;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}button,input,select{font:inherit}button,select,input[type=search]{border:1px solid var(--line);background:#172839;color:var(--text);border-radius:7px;padding:8px 11px}button{cursor:pointer;white-space:nowrap;font-weight:600}button:hover{border-color:var(--cyan);background:#243b4d}button:focus-visible,select:focus-visible,a:focus-visible,input:focus-visible{outline:2px solid var(--cyan);outline-offset:3px}button:disabled{opacity:.45;cursor:not-allowed}a{color:var(--cyan);text-underline-offset:3px}input[type=checkbox]{accent-color:var(--cyan);width:15px;height:15px;flex-shrink:0}input[type=range]{accent-color:var(--cyan);width:100%}h1,h2,h3,p{margin:0 0 12px}h1{font-size:25px;line-height:1.12;letter-spacing:-.7px;margin:3px 0 0;font-weight:700}h1 span{font-weight:400;color:var(--muted)}h2{font-size:19px;line-height:1.25;margin:0}h3{font-size:15px;margin-top:12px}table{width:100%;border-collapse:collapse;margin:12px 0;font-size:12px}th,td{text-align:left;border-bottom:1px solid #2a3d4e;padding:7px 3px;vertical-align:top}th{font-weight:500;color:var(--muted)}td{text-align:right;font-variant-numeric:tabular-nums;white-space:nowrap}.muted{color:var(--muted);font-size:12px}.eyebrow{color:var(--cyan);font-size:10px;letter-spacing:2px;font-weight:750}.accent{background:#69d5ff;color:#092131;border-color:#69d5ff}.accent:hover{background:#a0e4ff;color:#071c2a}.text-button{background:transparent;border:0;padding:3px;color:var(--cyan);font-size:12px}.brown{color:var(--orange)}.stephenson{color:var(--cyan)}.date-badge{font-size:12px;letter-spacing:1.5px;border-left:1px solid var(--line);padding:8px 18px;color:var(--muted)}
#app{height:100vh;height:100dvh;display:grid;grid-template-rows:82px minmax(0,1fr)}.masthead{display:flex;align-items:center;justify-content:space-between;padding:13px 24px;border-bottom:1px solid var(--line);background:var(--bg);gap:18px}.header-actions,.dialog-actions{display:flex;align-items:center;gap:9px}.workspace{display:grid;grid-template-columns:minmax(0,1fr) 345px;min-height:0}.map-shell{position:relative;min-width:0;min-height:0;overflow:hidden;background:#152a38}.map-canvas{position:absolute;inset:0;height:100%;width:100%}.sidebar{padding:18px;background:var(--panel);overflow-y:auto;overflow-x:hidden;overscroll-behavior:contain;scrollbar-color:#58748a #172839;scrollbar-width:thin;border-left:1px solid var(--line);min-height:0}.sidebar::-webkit-scrollbar{width:8px}.sidebar::-webkit-scrollbar-thumb{background:#58748a;border-radius:5px}.sidebar label{cursor:pointer}.selector-panel{padding-bottom:17px;border-bottom:1px solid var(--line);margin-bottom:18px}.selector-panel>label{font-size:12px;letter-spacing:.6px;font-weight:700}.select-row{display:flex;gap:7px;margin-top:8px}.select-row select{min-width:0;width:100%}.focus-row{display:flex;gap:6px;margin-top:9px}.focus-row button{font-size:11px;padding:7px}.sidebar-heading{display:flex;justify-content:space-between;align-items:center;gap:8px;margin-bottom:12px}.sidebar-heading h2{font-size:17px}.basemap-row{display:flex;flex-direction:column;gap:3px;margin-bottom:12px}.basemap-row label{display:flex;align-items:center;gap:7px}.basemap-row .muted{padding-left:25px}.scope-note{font-size:11px;color:#c1d0dc;line-height:1.6;padding:11px 12px;background:#1a2d3e;border-left:2px solid #69d5ff;border-radius:0 5px 5px 0}.sidebar-foot{padding-top:18px;font-size:11px;color:var(--muted)}.map-label{position:absolute;top:22px;left:65px;z-index:500;padding:12px 17px;background:#0b1420e8;border:1px solid #3f5368;border-radius:8px;box-shadow:0 6px 24px #0003;pointer-events:none;display:flex;flex-direction:column}.map-label strong{font-size:23px;letter-spacing:-.5px;margin:2px 0}.map-label>span:last-child{font-size:11px;color:var(--muted)}.map-message{position:absolute;z-index:700;top:125px;left:65px;right:30px;background:#263b4dee;border:1px solid var(--cyan);padding:12px;border-radius:7px;font-size:12px}.map-message[hidden]{display:none}
.layer-group{border-top:1px solid var(--line);margin-top:10px;padding-top:7px}.layer-group>summary{cursor:pointer;font-size:12px;font-weight:750;letter-spacing:.5px;padding:8px 0}.layer-row{border-radius:6px;padding:7px 0}.layer-row.is-active{background:#1c3042;padding:9px;margin:4px -9px}.layer-label{display:flex;gap:8px;align-items:flex-start;font-size:12px}.layer-label input{margin-top:3px}.layer-name{flex:1}.info-button{border:0;background:transparent;color:var(--cyan);padding:0 3px;font:italic 15px Georgia,serif;line-height:23px}.layer-detail{padding:10px 0 3px 23px;font-size:11px}.layer-detail p{line-height:1.5;margin:9px 0;color:var(--muted)}.legend-gradient{height:10px;border-radius:3px;border:1px solid #ffffff40}.legend-ticks{display:flex;justify-content:space-between;gap:4px;margin:5px 0 9px;font-size:9px;color:var(--muted)}.legend-ticks span:nth-child(2){text-align:center}.legend-ticks span:last-child{text-align:right}.swatch{display:inline-block;width:15px;height:8px;border-radius:2px;margin-right:7px;border:1px solid #ffffff40}.legend-category{font-size:11px;margin:5px 0}.opacity-label{display:flex;align-items:center;justify-content:space-between;font-size:10px;color:var(--muted)}.line-swatch{display:inline-block;width:22px;border-top:2px solid;margin:8px 5px 0 0;flex-shrink:0}.empty-note{color:var(--muted);font-size:12px;margin:14px 0}.source-link{font-size:11px}.layer-detail input[type=range]{height:15px;margin:5px 0}
.timeline{position:absolute;z-index:600;bottom:35px;left:22px;width:min(510px,calc(100% - 44px));padding:12px 15px;background:#0b1420ed;border:1px solid #40576a;border-radius:10px;box-shadow:0 8px 30px #0004}.timeline-head{display:flex;align-items:center;justify-content:space-between;margin-bottom:8px;gap:10px}.timeline-title{font-size:10px;letter-spacing:1.3px;font-weight:750;color:var(--muted)}.timeline-status{font-size:10px;color:var(--muted);min-height:15px;margin-top:6px}.timeline-main{display:flex;align-items:center;gap:12px}.timeline-main button{padding:5px 10px;min-width:70px;font-size:12px}.timeline-main input{flex:1}.timeline-year{font-size:21px;line-height:1;min-width:51px;font-weight:750;font-variant-numeric:tabular-nums}.timeline-dates{display:flex;justify-content:space-between;padding-left:83px;padding-right:66px;font-size:10px;color:var(--muted);margin-top:3px}.timeline .text-button{font-size:10px}.year-label{text-align:center}.timeline-hidden{opacity:.8}
.leaflet-container{background:#172d3d;font-family:inherit}.leaflet-control-zoom a{background:#152637;color:var(--text);border-color:#354a5e}.leaflet-control-zoom a:hover{background:#253b50}.leaflet-control-attribution{background:#0b1420dd!important;color:#bcd0de!important;font-size:9px!important;max-width:90%}.leaflet-control-attribution a{color:#8ddcff}.leaflet-control-scale-line{background:#0b1420c9;border-color:#d5e3ee;color:#e8eff5;text-shadow:none}.leaflet-popup-content-wrapper,.leaflet-popup-tip{background:#122434;color:var(--text)}.leaflet-popup-content{line-height:1.5;font-size:12px;max-height:230px;overflow:auto}.leaflet-tooltip{background:#0b1420ed;color:var(--text);border:1px solid #455e74;box-shadow:0 4px 18px #0004}.glacier-tip strong{display:block;font-size:14px}.glacier-tip span{color:var(--muted);font-size:10px}.glacier-name-label{background:#102233d9;color:#e8eff5;border:1px solid #7d92a088;box-shadow:none;font-size:10px;font-weight:650;letter-spacing:.3px;padding:2px 5px}.glacier-name-label::before{display:none}
.large-dialog,.credits-dialog{padding:0;border:1px solid #4d6478;border-radius:12px;background:var(--bg);color:var(--text);width:min(1500px,96vw);height:min(1000px,94vh);max-width:96vw;max-height:94vh;overflow:hidden;box-shadow:0 28px 110px #000b}.large-dialog::backdrop,.credits-dialog::backdrop{background:#050d17b8;backdrop-filter:blur(5px)}.dialog-shell{height:100%;display:flex;flex-direction:column;min-height:0}.dialog-head{display:flex;justify-content:space-between;align-items:center;padding:16px 22px;gap:15px;border-bottom:1px solid var(--line);flex-shrink:0}.dialog-head h2{font-size:25px;margin-top:2px}.close{font-size:25px;font-weight:400;line-height:1;padding:4px 9px;min-width:35px}.tabs{display:flex;gap:3px;padding:0 17px;background:#112232;border-bottom:1px solid var(--line);overflow-x:auto;flex-shrink:0}.tabs button{border:0;border-radius:0;background:none;color:var(--muted);padding:13px 11px;font-size:12px}.tabs button[aria-selected=true]{color:var(--cyan);border-bottom:2px solid var(--cyan);background:#1b3143}.dashboard-grid{display:grid;grid-template-columns:minmax(0,1fr) 340px;flex:1;min-height:0}.dashboard-grid .sidebar{font-size:12px}.dashboard-grid .sidebar h3{font-size:14px}.dashboard-grid .sidebar p{color:var(--muted)}#active-layer-count{font-size:11px;color:var(--muted)}#glacier-notes{border-bottom:1px solid var(--line);padding-bottom:10px;margin-bottom:10px}.terrain-grid{display:grid;grid-template-columns:minmax(0,1fr) 290px;flex:1;min-height:0}.terrain-controls>label{display:block;margin:15px 0 6px;font-size:12px;font-weight:650}.terrain-controls select{width:100%;font-size:12px}.terrain-controls .check-row{display:flex;align-items:center;gap:8px}.terrain-controls output{float:right;color:var(--cyan)}.instruction{font-size:11px;color:var(--muted)}#terrain-plot{position:relative;min-height:0;min-width:0;height:100%;background:#0b1420}.terrain-timeline{padding:14px 0}.terrain-timeline .timeline-main{gap:8px}.terrain-timeline .timeline-dates{padding-left:0;padding-right:0}.terrain-timeline .timeline-year{font-size:18px}.terrain-timeline .timeline-main button{min-width:58px}.terrain-note{font-size:11px;border-top:1px solid var(--line);padding-top:16px;color:var(--muted)}.terrain-note strong{color:var(--text)}.terrain-note p{margin:8px 0}.terrain-controls small{font-size:10px;color:var(--muted)}#terrain-legend{margin:15px 0}#terrain-legend p{font-size:11px;color:var(--muted)}.plot-error{padding:40px;max-width:650px;color:var(--muted)}
.credits-dialog{width:min(1050px,94vw);height:min(900px,90vh)}.credits-content{padding:22px;overflow-y:auto;min-height:0;overscroll-behavior:contain}.credits-content>p{color:var(--muted);max-width:800px}.credits-tools{display:flex;flex-wrap:wrap;gap:12px;align-items:center;margin:20px 0;font-size:12px}.credits-tools input{flex:1;min-width:200px}.source-card{padding:15px 0;border-top:1px solid var(--line)}.source-card h3{font-size:15px;margin:0 0 8px}.source-card p{font-size:12px;color:var(--muted)}.status-badge{font-size:10px;padding:3px 6px;background:#57482c;color:#ffdc95;border-radius:4px;display:inline-block;margin-bottom:8px}.source-card details{font-size:11px}.credits-content summary{cursor:pointer;padding:12px 0;font-weight:650}.credits-content pre{white-space:pre-wrap;word-break:break-word;font-size:10px;padding:12px;background:#152738;border-radius:5px;max-height:350px;overflow:auto}.credits-content details>p{font-size:12px;color:var(--muted)}.credits-content details>div{font-size:12px}.credit-layer{padding:9px 0;border-top:1px solid var(--line)}.credit-layer span{display:block;color:var(--muted);font-size:11px}.credit-layer code{word-break:break-word;font-size:10px}.loading{display:flex;align-items:center;justify-content:center;width:100%;height:100%;min-height:200px;color:var(--muted)}#toast{position:fixed;z-index:2000;left:50%;bottom:22px;transform:translateX(-50%);max-width:75vw;padding:12px 18px;border:1px solid var(--cyan);border-radius:8px;background:#173247;box-shadow:0 10px 45px #0009;font-size:13px}#toast[hidden]{display:none}
@media(min-width:1700px){.workspace{grid-template-columns:minmax(0,1fr) 365px}.dashboard-grid{grid-template-columns:minmax(0,1fr) 365px}}
@media(max-width:950px){.masthead{padding:10px 16px}.date-badge{display:none}.workspace{grid-template-columns:minmax(0,1fr) 300px}.sidebar{padding:14px}.header-actions button{font-size:12px}.dashboard-grid{grid-template-columns:minmax(0,1fr) 290px}.terrain-grid{grid-template-columns:minmax(0,1fr) 255px}.map-label{left:60px;top:14px;padding:9px 12px}.map-label strong{font-size:19px}.map-label>span:last-child{display:none}.dialog-head{padding:12px 16px}.dialog-head h2{font-size:21px}.timeline-dates{padding-right:62px}.header-actions{gap:5px}}
@media(max-width:650px){#app{grid-template-rows:105px minmax(0,1fr)}.masthead{align-items:flex-start;flex-direction:column;gap:8px}h1{font-size:20px}.brand{display:flex;align-items:center;gap:12px}.brand .eyebrow{max-width:85px;font-size:9px}.header-actions{width:100%}.header-actions button{padding:5px 8px;font-size:11px}.workspace{grid-template-columns:1fr;grid-template-rows:minmax(270px,58%) minmax(0,42%)}.sidebar{border-left:0;border-top:1px solid var(--line)}.selector-panel{margin-bottom:10px;padding-bottom:10px}.map-label{display:none}.timeline{bottom:28px;left:10px;width:calc(100% - 20px);padding:8px 12px}.timeline-status{font-size:9px}.timeline-title{font-size:9px}.timeline-main{gap:8px}.timeline-year{font-size:18px}.timeline-main button{min-width:59px}.timeline-dates{padding-left:67px;padding-right:62px}.large-dialog{height:97vh;max-height:97vh;width:98vw;max-width:98vw;border-radius:6px}.dialog-head{padding:10px;gap:8px}.dialog-head h2{font-size:18px}.dialog-actions{gap:5px}.dialog-actions button{font-size:10px;padding:6px}.dialog-actions .close{font-size:24px;padding:1px 7px}.dialog-head .eyebrow{font-size:8px}.dashboard-grid,.terrain-grid{grid-template-columns:1fr;grid-template-rows:minmax(250px,55%) minmax(0,45%)}.tabs{padding:0 5px}.tabs button{font-size:10px;padding:10px 8px}.terrain-controls{padding:12px}.terrain-controls>label{margin-top:9px}.credits-content{padding:14px}.credits-tools{gap:9px}.credits-dialog .dialog-head h2{font-size:19px}}
/* Preserve the display grid of scientific rasters when zooming. */
.leaflet-image-layer{image-rendering:pixelated}

/* The floating animation panel can shrink without removing the selected footprint. */
.timeline-head{margin-bottom:0}.timeline-title{flex:1;min-width:0}
.timeline-tools{display:flex;justify-content:flex-end;margin:4px 0 6px}
.timeline-collapse{flex-shrink:0;padding:5px 2px}
.timeline-compact-date{color:var(--cyan);font-size:12px;font-weight:700;font-variant-numeric:tabular-nums;white-space:nowrap}
.timeline-body[hidden],.timeline-compact-date[hidden]{display:none}
.timeline.is-collapsed{width:min(430px,calc(100% - 44px));padding:7px 12px}
.glacier-name-label{pointer-events:none}
/* Dim the island behind dialogs; glacier imagery is also dimmed in the map setup. */
.large-dialog::backdrop,.credits-dialog::backdrop{background:rgba(5,13,23,.78)}
@media(max-width:650px){.timeline.is-collapsed{width:calc(100% - 20px);padding:6px 10px}.timeline-head{gap:7px}.timeline.is-collapsed .timeline-title{letter-spacing:.8px}}

'''


In [ ]:
## Layer controls, lazy glacier loading, animation, source modal and 3D ##

# Connect browser controls, label placement and fullscreen fallback.
APP_JS = r'''

/* Heard Island explorer. Local assets are lazy-loaded; no Python is needed after export. */
(() => {
'use strict';
const H = window.HI, D = H.overview, $ = id => document.getElementById(id);
const esc = value => String(value ?? '').replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const years = [1947,1988,2014,2019,2026];
const colours = D.year_colours;
const focus = ['Brown','Stephenson'];
let toastTimer, island, glacierView = null, currentGlacier = null, terrainState = null, glacierRequest = 0;
function toast(message) { $('toast').textContent = message; $('toast').hidden = false; clearTimeout(toastTimer); toastTimer = setTimeout(() => $('toast').hidden = true, 5500); }
function showError(message) { $('map-message').textContent = message; $('map-message').hidden = false; }
if (!window.L) { showError('Leaflet could not load. Re-run the export with internet access, or connect to load the map library.'); return; }
const scriptPromises = new Map();
function loadScript(path) {
  if (scriptPromises.has(path)) return scriptPromises.get(path);
  const promise = new Promise((resolve,reject) => {
    const script = document.createElement('script'); script.src = path + (path.includes('?') ? '&' : '?') + 'v=' + D.build_id;
    script.onload = resolve; script.onerror = () => { scriptPromises.delete(path); script.remove(); reject(new Error('Could not load ' + path + '. Keep the assets folder beside the HTML.')); };
    document.head.appendChild(script);
  }); scriptPromises.set(path,promise); return promise;
}
function preventMapScroll(element) {
  // Stop propagation, not default behaviour: the panel must still scroll normally.
  L.DomEvent.disableClickPropagation(element); L.DomEvent.disableScrollPropagation(element);
  element.addEventListener('wheel', e => e.stopPropagation(), {passive:true});
  element.addEventListener('touchmove', e => e.stopPropagation(), {passive:true});
}
document.querySelectorAll('.sidebar,.timeline,.credits-content').forEach(preventMapScroll);
function sourceButtons(ids) {
  return (ids || []).map(id => `<button class="text-button source-link" data-source="${esc(id)}">${esc(D.sources[id]?.title || id)} ↗</button>`).join('<br>');
}
function legend(spec) {
  if (!spec) return '';
  let scale = spec.discrete ? spec.colours.map((c,i) => `<div class="legend-category"><span class="swatch" style="background:${c}"></span>${esc(spec.ticks[i])}</div>`).join('') :
    `<div class="legend-gradient" style="background:linear-gradient(to right,${spec.colours.join(',')})"></div><div class="legend-ticks">${spec.ticks.map(t=>`<span>${esc(t)}</span>`).join('')}</div>`;
  return scale + `<p>${esc(spec.note)}</p>`;
}
function vectorLegend(layer) {
  if (layer.id === 'geology') return Object.entries(D.rock_colours).map(([label,colour]) => `<div class="legend-category"><span class="swatch" style="background:${colour}"></span>${esc(label)}</div>`).join('') + '<p>Mapped surface units; Snow/Ice does not identify the substrate beneath ice.</p>';
  if (layer.id.startsWith('ice_')) return '';
  if (layer.id === 'lost_1947_2026') return '<p>Filled area = inside the 1947 outline and outside the 2026 outline. Remaining 2026 ice is excluded.</p><div><span class="swatch" style="background:#ffb36b"></span>Brown</div><div><span class="swatch" style="background:#69d5ff"></span>Stephenson</div>';
  if (layer.id === 'retreat_zones') return ['1947–1988','1988–2014','2014–2019','2019–2026','2026 remaining'].map((label,i)=>`<div class="legend-category"><span class="swatch" style="background:${['#F4DF70','#BE9AFF','#FF83AF','#8FE88C','#E8EFF5'][i]}"></span>${label}</div>`).join('')+'<p>Separate mapped loss intervals, plus ice remaining in 2026. These polygons are context for the geology comparison.</p>';
  if (layer.id === 'coastline') return '<p>Line boundary of the supplied coastline polygon.</p>';
  if (layer.id === 'lagoons') return '<p>Filled lagoon extents from the supplied 2014 polygons; not measured water level.</p>';
  if (layer.id === 'bed_highs') return '<p>Screened modelled bed highs. Retained-trial counts are not confidence probabilities.</p>';
  if (layer.tab === 'Field evidence') return '<p>Survey/archive evidence. Depths are relative to survey water. Dashed geometry marks provisional registration.</p>';
  return '<p>Click features for their recorded attributes.</p>';
}
function outlineStyle(feature, fill = false) {
  const p = feature.properties || {}, year = p.year;
  return {color:colours[year] || '#e8eff5',weight:2.1,opacity:1,fillColor:colours[year] || '#e8eff5',fillOpacity:fill?.22:0,
    dashArray:p.provisional?'2 6':({'1947':'8 4','1988':'5 4','2014':'','2019':'4 3','2026':''}[year] || '')};
}
function styleVector(feature,id) {
  const p = feature.properties || {};
  let colour = D.glacier_colours[p.name_1] || '#D8E3EC';
  if (id === 'retreat_zones') colour = {'1947–1988':'#F4DF70','1988–2014':'#BE9AFF','2014–2019':'#FF83AF','2019–2026':'#8FE88C','2026 remaining':'#E8EFF5'}[p.zone] || colour;
  if (id === 'geology') colour = D.rock_colours[p.Rock_Type || 'Unknown'] || '#BCCBD7';
  if (id === 'lagoons' || id.includes('lagoon')) colour = '#69D5FF';
  if (id === 'coastline') return {color:'#d1ddd9',weight:1.2,opacity:.9,fill:false};
  const provisional = String(p.geometry_status || '').startsWith('provisional');
  return {color:provisional?'#F4DF70':colour,weight:id==='transects'?3:1.5,fillColor:colour,
    fillOpacity:id==='geology'?.55:id==='lost_1947_2026'?.55:.4,opacity:1,
    dashArray:provisional || id==='cross_sections'?'6 4':''};
}
function featureInfo(feature) {
  const p = feature.properties || {};
  let html = '';
  Object.entries(D.tooltip_labels).forEach(([key,label]) => {
    if (p[key] != null) html += `<tr><th>${esc(label)}</th><td>${esc(typeof p[key] === 'number' ? Number(p[key].toFixed(2)) : p[key])}</td></tr>`;
  });
  return html ? '<table>' + html + '</table>' : '';
}
class View {
  constructor(mapId,history,bounds) {
    this.map = L.map(mapId,{zoomControl:true,preferCanvas:true,minZoom:7,maxZoom:18,zoomSnap:.25});
    this.map.createPane('rasterData').style.zIndex = 350;
    this.map.createPane('polygons').style.zIndex = 380;
    this.map.createPane('inventory').style.zIndex = 420;
    this.map.createPane('evidence').style.zIndex = 460;
    this.map.createPane('names').style.zIndex = 490;
    this.tiles = L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',{
      // Dim imagery inside glacier dashboards so evidence layers remain legible.
      attribution:esc(D.imagery_credit),maxZoom:19,crossOrigin:true,
      opacity:mapId==='glacier-map'?.48:1}).addTo(this.map);
    this.map.attributionControl.setPrefix('<a href="https://leafletjs.com/" target="_blank" rel="noopener">Leaflet</a>');
    L.control.scale({imperial:false,position:'bottomleft',maxWidth:140}).addTo(this.map);
    this.bounds = bounds; this.map.fitBounds(bounds,{padding:[35,35]}); this.history = history;
    this.active = new Map(); this.definitions = new Map(); this.rowContainers = new Set(); this.timeline = null;
  }
  layerObject(layer) {
    if (layer.type === 'raster') return L.imageOverlay(layer.image,layer.bounds,{opacity:.78,pane:'rasterData',interactive:false,alt:layer.title});
    const isIce = layer.id.startsWith('ice_');
    return L.geoJSON(layer.data,{
      pane:isIce?'inventory':(['geology','lagoons','coastline'].includes(layer.id)?'polygons':'evidence'),
      style:f => isIce?outlineStyle(f):styleVector(f,layer.id),
      pointToLayer:(f,ll) => L.circleMarker(ll,{radius:5,pane:'evidence',weight:1.3,fillOpacity:.9,...styleVector(f,layer.id)}),
      onEachFeature:(f,obj) => {
        if (isIce) {
          const p = f.properties;
          obj.bindTooltip(`${esc(p.name_1)} · ${p.year}${p.provisional?' · provisional':''}`);
          obj.on('click',()=>p.RGIId && openGlacier(p.RGIId));
        } else {
          const info = featureInfo(f); if (info) obj.bindPopup(info + sourceButtons(layer.sources),{maxWidth:310});
        }
      }
    });
  }
  add(layer) {
    if (this.active.has(layer.id)) return;
    const object = this.layerObject(layer); object.addTo(this.map); this.active.set(layer.id,{layer,object,opacity:.78}); this.sync();
  }
  remove(id) { const entry = this.active.get(id); if (entry) this.map.removeLayer(entry.object); this.active.delete(id); this.sync(); }
  clear() { for (const id of [...this.active.keys()]) this.remove(id); if(this.timeline)this.timeline.hide(); this.sync(); }
  opacity(id,value) {
    const entry = this.active.get(id); if (!entry) return; entry.opacity = value;
    if (entry.layer.type === 'raster') entry.object.setOpacity(value);
    else entry.object.eachLayer(obj => obj.setStyle && obj.setStyle({opacity:value,fillOpacity:value*.65}));
  }
  sync() {
    for (const container of this.rowContainers) container.querySelectorAll('[data-layer]').forEach(row=>{
      const active = this.active.has(row.dataset.layer), input = row.querySelector('input[type=checkbox]');
      input.checked=active; row.classList.toggle('is-active',active); row.querySelector('.layer-detail').hidden=!active;
      if(active){const percentage=Math.round(this.active.get(row.dataset.layer).opacity*100);row.querySelector('input[type=range]').value=percentage;row.querySelector('output').textContent=percentage+'%';}
    });
    if (this === glacierView) $('active-layer-count').textContent=`${this.active.size} active layer${this.active.size===1?'':'s'}`;
  }
  destroy() { if(this.timeline)this.timeline.destroy(); this.map.remove(); this.active.clear(); this.rowContainers.clear(); }
}
function inventoryLayers(history) {
  return years.filter(year => history.some(f=>f.properties.year===year)).map(year=>({
    id:'ice_'+year,title:`${year} inventory`,tab:'Retreat',type:'vector',
    year,data:{type:'FeatureCollection',features:history.filter(f=>f.properties.year===year)},sources:D.inventory_sources[year] || []
  }));
}
function renderLayerRows(container,layers,view,grouped=false) {
  container.replaceChildren(); view.rowContainers.add(container);
  if (!layers.length) { container.innerHTML='<p class="empty-note">No saved spatial layer is available for this section.</p>'; return; }
  const groups = grouped ? [...new Set(layers.map(l=>l.tab))] : [''];
  for(const group of groups) {
    const subset = grouped?layers.filter(l=>l.tab===group):layers;
    let holder = container;
    if(grouped) { holder = document.createElement('details'); holder.className='layer-group'; holder.open=group==='Retreat'; holder.innerHTML=`<summary>${esc(group==='Overview'?'Context':group)} <span class="muted">· ${subset.length}</span></summary>`; container.append(holder); }
    subset.forEach(layer=>{
      view.definitions.set(layer.id,layer);
      const row = document.createElement('div'); row.className='layer-row'; row.dataset.layer=layer.id;
      const swatch=layer.year?`<span class="line-swatch" style="border-color:${colours[layer.year]}"></span>`:'';
      row.innerHTML=`<div class="layer-label"><label class="layer-label layer-name"><input type="checkbox" aria-label="${esc(layer.title)}">${swatch}<span>${esc(layer.title)}</span></label><button class="info-button" title="Layer sources and metadata" aria-label="Sources for ${esc(layer.title)}">i</button></div><div class="layer-detail" hidden>${layer.type==='raster'?legend(layer.style):vectorLegend(layer)}<div class="opacity-label"><span>Opacity</span><output>78%</output></div><input type="range" min="0" max="100" value="78" aria-label="${esc(layer.title)} opacity"><button class="text-button layer-source">Source & metadata ↗</button></div>`;
      row.querySelector('input[type=checkbox]').addEventListener('change',e=> e.target.checked?view.add(layer):view.remove(layer.id));
      row.querySelector('input[type=range]').addEventListener('input',e=>{ row.querySelector('output').textContent=e.target.value+'%'; view.opacity(layer.id,Number(e.target.value)/100); });
      row.querySelectorAll('.info-button,.layer-source').forEach(b=>b.addEventListener('click',()=>openCredits(layer.sources,layer)));
      holder.append(row);
    });
  } view.sync();
}
class Timeline {
  constructor(container,view) {
    this.container=container;this.view=view;this.index=2;this.visible=false;this.playing=false;this.timer=null;this.overlay=null;
    this.collapsed=false;
    const bodyId=container.id+'-controls';
    container.innerHTML='<div class="timeline-head"><span class="timeline-title">GLACIER RETREAT · MAPPED DATES</span><span class="timeline-compact-date" hidden>No date shown</span><button class="text-button timeline-collapse" aria-expanded="true" aria-controls="'+bodyId+'" aria-label="Collapse animation controls">Collapse ▾</button></div><div id="'+bodyId+'" class="timeline-body"><div class="timeline-tools"><button class="text-button timeline-toggle">Show date</button></div><div class="timeline-main"><button class="play">▶ Play</button><input type="range" min="0" max="4" value="2" step="1" aria-label="Inventory date"><output class="timeline-year">2014</output></div><div class="timeline-dates">'+years.map(y=>`<span>${y}</span>`).join('')+'</div><div class="timeline-status">Step through mapped dates; intervals are not equally spaced in time.</div></div>';
    container.querySelector('.play').onclick=()=>this.playing?this.pause():this.play();
    container.querySelector('input').oninput=e=>{this.pause();this.index=Number(e.target.value);this.show();};
    container.querySelector('.timeline-toggle').onclick=()=>this.visible?this.hide():this.show();
    container.querySelector('.timeline-collapse').onclick=()=>this.setCollapsed(!this.collapsed);
  }
  setCollapsed(collapsed) {
    // Collapse pauses playback but preserves the selected footprint on the map.
    if(collapsed)this.pause();
    this.collapsed=collapsed;this.container.classList.toggle('is-collapsed',collapsed);
    this.container.querySelector('.timeline-body').hidden=collapsed;
    this.container.querySelector('.timeline-compact-date').hidden=!collapsed;
    const button=this.container.querySelector('.timeline-collapse');
    button.textContent=collapsed?'Expand ▴':'Collapse ▾';
    button.setAttribute('aria-expanded',String(!collapsed));
    button.setAttribute('aria-label',collapsed?'Expand animation controls':'Collapse animation controls');
    if(this.view===island)requestAnimationFrame(layoutIslandLabels);
  }
  show() {
    this.visible=true;
    // A single animated date replaces selected static inventory layers, avoiding ambiguous overlaps.
    for(const id of [...this.view.active.keys()])if(id.startsWith('ice_'))this.view.remove(id);
    if(this.overlay)this.view.map.removeLayer(this.overlay);
    const year=years[this.index], features=this.view.history.filter(f=>f.properties.year===year);
    this.overlay=L.geoJSON({type:'FeatureCollection',features},{pane:'inventory',interactive:false,style:f=>outlineStyle(f,true)}).addTo(this.view.map);
    this.container.querySelector('.timeline-toggle').textContent='Hide date';
    this.container.querySelector('.timeline-year').textContent=year;
    this.container.querySelector('.timeline-compact-date').textContent=String(year);
    this.container.querySelector('input').value=this.index;
    const provisional=features.some(f=>f.properties.provisional);
    this.container.querySelector('.timeline-status').textContent=!features.length?'No saved outline for this date.':provisional?'Includes provisional recent outlines · dotted boundaries.':year===1947?'1947 is nominal; some records pre-date this year.':'Mapped inventory · equal screen time, unequal real intervals.';
  }
  hide() {this.pause();this.visible=false;if(this.overlay)this.view.map.removeLayer(this.overlay);this.overlay=null;this.container.querySelector('.timeline-toggle').textContent='Show date';this.container.querySelector('.timeline-compact-date').textContent='No date shown';}
  play() {if(!this.visible)this.index=0;this.playing=true;this.container.querySelector('.play').textContent='Ⅱ Pause';this.show();this.timer=setInterval(()=>{this.index=(this.index+1)%years.length;this.show();},1350);}
  pause() {clearInterval(this.timer);this.timer=null;this.playing=false;this.container.querySelector('.play').textContent='▶ Play';}
  destroy(){this.hide();this.container.replaceChildren();}
}
// Island map and the all-glacier inventory controls.
island=new View('island-map',D.history,D.bounds);island.timeline=new Timeline($('island-timeline'),island);
const selection=L.geoJSON({type:'FeatureCollection',features:D.history.filter(f=>f.properties.year===2014 && f.properties.RGIId)},{
  pane:'inventory',style:f=>({color:D.glacier_colours[f.properties.name_1]||'#d6e4ec',weight:1,opacity:.5,fillOpacity:.025}),
  onEachFeature:(f,layer)=>{
    layer.bindTooltip(`<div class="glacier-tip"><strong>${esc(f.properties.name_1)}</strong><span>Open glacier dashboard ↗</span></div>`,{sticky:true});
    layer.on('click',()=>openGlacier(f.properties.RGIId));
    layer.on('mouseover',()=>layer.setStyle({weight:2,fillOpacity:.12}));
    layer.on('mouseout',()=>layer.setStyle({weight:1,fillOpacity:.025}));
  }
}).addTo(island.map);
const nameLabels=L.layerGroup().addTo(island.map);
const nameLabelEntries=[];
D.glaciers.forEach(g=>{
  // Use a point within the glacier footprint, rather than its buffered bounding box.
  const anchor=g.label_anchor?L.latLng(g.label_anchor):L.latLngBounds(g.bounds).getCenter();
  const name=String(g.name||'').trim()||g.id;
  const marker=L.marker(anchor,{interactive:false,icon:L.divIcon({className:'',html:'',iconSize:[0,0]}),pane:'names'})
    .bindTooltip(esc(name),{permanent:true,direction:'center',className:'glacier-name-label',pane:'names'}).addTo(nameLabels);
  const leader=L.polyline([],{pane:'names',interactive:false,color:'#b3c5d3',weight:1,opacity:.65}).addTo(nameLabels);
  nameLabelEntries.push({marker,leader,anchor,focus:g.focus});
  const option=document.createElement('option');option.value=g.id;option.textContent=name; $('glacier-select').append(option);
});
function layoutIslandLabels(){
  // Keep all available names; move colliding labels and connect them to their glacier.
  const map=island.map,size=map.getSize(),mapRect=map.getContainer().getBoundingClientRect();
  const used=[];
  for(const element of map.getContainer().parentElement.querySelectorAll('.map-label,.timeline')){
    const r=element.getBoundingClientRect();
    if(r.width&&r.height)used.push({left:r.left-mapRect.left-6,right:r.right-mapRect.left+6,top:r.top-mapRect.top-6,bottom:r.bottom-mapRect.top+6});
  }
  const overlap=(a,b)=>Math.max(0,Math.min(a.right,b.right)-Math.max(a.left,b.left))*Math.max(0,Math.min(a.bottom,b.bottom)-Math.max(a.top,b.top));
  const candidates=[[0,0]];
  for(const radius of [25,50,80,115,155,200])for(let i=0;i<16;i++){
    const angle=i*Math.PI/8;candidates.push([radius*Math.cos(angle),radius*Math.sin(angle)]);
  }
  for(const entry of [...nameLabelEntries].sort((a,b)=>Number(b.focus)-Number(a.focus))){
    const label=entry.marker.getTooltip().getElement();if(!label)continue;
    const anchor=map.latLngToContainerPoint(entry.anchor);
    const onScreen=anchor.x>=0&&anchor.y>=0&&anchor.x<=size.x&&anchor.y<=size.y;
    label.style.visibility=onScreen?'visible':'hidden';entry.leader.setLatLngs([]);if(!onScreen)continue;
    const width=label.offsetWidth+8,height=label.offsetHeight+6;let best=null;
    for(const [dx,dy] of candidates){
      const x=Math.max(width/2+8,Math.min(size.x-width/2-8,anchor.x+dx));
      const y=Math.max(height/2+8,Math.min(size.y-height/2-8,anchor.y+dy));
      const box={left:x-width/2,right:x+width/2,top:y-height/2,bottom:y+height/2};
      const cost=used.reduce((n,r)=>n+overlap(box,r),0)*1000+Math.hypot(x-anchor.x,y-anchor.y);
      if(!best||cost<best.cost)best={x,y,box,cost};
      if(cost===0)break;
    }
    used.push(best.box);const position=map.containerPointToLatLng([best.x,best.y]);
    entry.marker.setLatLng(position);
    if(Math.hypot(best.x-anchor.x,best.y-anchor.y)>18)entry.leader.setLatLngs([entry.anchor,position]);
  }
}
island.map.on('moveend zoomend resize',()=>requestAnimationFrame(layoutIslandLabels));
requestAnimationFrame(layoutIslandLabels);
renderLayerRows($('island-layers'),[...inventoryLayers(D.history),...D.layers],island,true);
$('island-clear').onclick=()=>island.clear();
$('island-reset').onclick=()=>island.map.fitBounds(D.bounds,{padding:[35,35]});
$('island-imagery').onchange=e=>e.target.checked?island.tiles.addTo(island.map):island.map.removeLayer(island.tiles);
$('glacier-open').onclick=()=>{if($('glacier-select').value)openGlacier($('glacier-select').value);else $('glacier-select').focus();};
$('glacier-select').onchange=()=>{const g=D.glaciers.find(g=>g.id===$('glacier-select').value);if(g)island.map.fitBounds(g.bounds,{padding:[45,45]});};
document.querySelectorAll('[data-glacier]').forEach(b=>b.onclick=()=>openGlacier(b.dataset.glacier));
function resizeViews(){island.map.invalidateSize();if(glacierView)glacierView.map.invalidateSize();if(terrainState&&window.Plotly&&$('terrain-plot')._fullLayout)Plotly.Plots.resize($('terrain-plot'));}
function syncFullscreenButton(){
  const active=Boolean(document.fullscreenElement||document.webkitFullscreenElement);
  const button=$('fullscreen');button.setAttribute('aria-pressed',String(active));
  const fallback=!document.documentElement.requestFullscreen&&!document.documentElement.webkitRequestFullscreen||document.fullscreenEnabled===false;
  const label=active?'Exit full screen':fallback?'Open map in a full window':'Enter full screen';
  button.title=label;button.setAttribute('aria-label',label);requestAnimationFrame(resizeViews);
}
function openFullWindow(){
  // Embedded notebook policies can deny fullscreen; the exported page remains usable.
  const link=document.createElement('a');link.href=window.location.href;link.target='_blank';link.rel='noopener';
  document.body.append(link);link.click();link.remove();
}
$('fullscreen').onclick=async()=>{
  try{
    if(document.fullscreenElement)await document.exitFullscreen();
    else if(document.webkitFullscreenElement)document.webkitExitFullscreen();
    else if(document.fullscreenEnabled===false)openFullWindow();
    else if(document.documentElement.requestFullscreen)await document.documentElement.requestFullscreen();
    else if(document.documentElement.webkitRequestFullscreen)document.documentElement.webkitRequestFullscreen();
    else openFullWindow();
  }catch(error){openFullWindow();}
};
document.addEventListener('fullscreenchange',syncFullscreenButton);
document.addEventListener('webkitfullscreenchange',syncFullscreenButton);
syncFullscreenButton();

async function openGlacier(id) {
  const definition=D.glaciers.find(g=>g.id===id);if(!definition)return;
  const request=++glacierRequest;toast('Opening '+definition.name+'…');
  try {if(!H.glaciers[id])await loadScript(definition.data);if(request!==glacierRequest)return;
    const data=H.glaciers[id];if(!data)throw new Error('Glacier data did not initialise.');
    if(glacierView){glacierView.destroy();glacierView=null;}
    island.timeline.pause();currentGlacier=id;$('glacier-title').textContent=data.name;
    if(!$('glacier-dialog').open)$('glacier-dialog').showModal();
    glacierView=new View('glacier-map',data.history,data.bounds);
    glacierView.timeline=new Timeline($('glacier-timeline'),glacierView);
    const tabs=$('glacier-tabs');tabs.replaceChildren();
    data.tabs.forEach(tab=>{const button=document.createElement('button');button.textContent=tab;button.setAttribute('role','tab');button.onclick=()=>selectTab(tab);button.dataset.tab=tab;tabs.append(button);});
    selectTab('Overview');
    // The most recent saved boundary is a useful opening context, with status retained.
    const latest=Math.max(...data.history.map(f=>f.properties.year));
    glacierView.add(inventoryLayers(data.history).find(l=>l.year===latest));
    $('glacier-clear').onclick=()=>glacierView.clear();
    $('glacier-3d').onclick=()=>openTerrain(id);
    $('glacier-sources').onclick=()=>openCredits([...new Set(data.layers.flatMap(l=>l.sources))]);
    requestAnimationFrame(()=>{glacierView.map.invalidateSize();glacierView.map.fitBounds(data.bounds,{padding:[30,30]});});
    $('toast').hidden=true;
  }catch(error){toast(error.message);}
}
function selectTab(tab){
  const data=H.glaciers[currentGlacier];if(!data||!glacierView)return;
  $('glacier-tabs').querySelectorAll('button').forEach(b=>b.setAttribute('aria-selected',String(b.dataset.tab===tab)));
  $('glacier-notes').innerHTML=data.notes[tab]||'';
  const layers=tab==='Retreat'?[...inventoryLayers(data.history),...data.layers.filter(l=>l.tab===tab)]:data.layers.filter(l=>l.tab===tab);
  renderLayerRows($('glacier-layers'),layers,glacierView);$('glacier-sidebar').scrollTop=0;
}
$('glacier-dialog').addEventListener('close',()=>{glacierRequest++;if(glacierView){glacierView.destroy();glacierView=null;}currentGlacier=null;});

// Source register: bibliography, local lineage, availability and technical references.
let creditsFilter=null, creditsLayer=null;
function openCredits(ids=null,layer=null){creditsFilter=ids?new Set(ids):null;creditsLayer=layer;$('credits-search').value='';renderCredits();if(!$('credits-dialog').open)$('credits-dialog').showModal();}
function renderCredits(){
  const query=$('credits-search').value.toLowerCase();
  $('credits-list').innerHTML=(creditsLayer?`<div class="scope-note"><strong>${esc(creditsLayer.title)}</strong><br>${esc(creditsLayer.path||'Inventory geometry')}<br>${creditsLayer.native_grid?esc('Native grid: '+creditsLayer.native_grid.crs+' · '+creditsLayer.native_grid.pixel_size.join(' × ')+' units per cell'):''}</div>`:'')+
    Object.values(D.sources).filter(s=>(!creditsFilter||creditsFilter.has(s.id))&&(!query||JSON.stringify(s).toLowerCase().includes(query))).map(s=>{
      const unresolved=s.citation_status!=='verified';
      const documentation=s.metadata?.documentation_download;
      return `<article class="source-card"><h3>${esc(s.title)}</h3>${unresolved?`<span class="status-badge">${esc(s.citation_status.replaceAll('_',' '))}</span>`:''}<p>${esc(s.citation||'Bibliographic details unresolved.')}</p>${s.url?`<p><a href="${esc(s.url)}" target="_blank" rel="noopener">Source record / DOI ↗</a></p>`:''}<p>Licence: ${esc(s.licence||'See source documentation; no licence is assumed.')}</p>${documentation?`<p><a href="${esc(documentation)}" target="_blank" rel="noopener">Supplied source documentation ↗</a></p>`:''}<details><summary>Metadata & processing context</summary><pre>${esc(JSON.stringify(s.metadata,null,2))}</pre></details></article>`;
    }).join('');
  $('technical-references').innerHTML='<ul>'+D.technical_references.map(r=>`<li><a href="${esc(r.url)}" target="_blank" rel="noopener">${esc(r.title)}</a></li>`).join('')+'</ul>';
  $('credits-layers').innerHTML=D.catalogue.filter(r=>(!query||JSON.stringify(r).toLowerCase().includes(query))).map(r=>`<div class="credit-layer"><strong>${esc(r.title)}</strong><span>${esc(r.glacier_name||r.glacier_id||'Island / shared')} · ${r.available?'File found':'File unavailable at build'}${r.display_error?' · display failed':''}</span><code>${esc(r.path)}</code><span>Sources: ${esc(r.sources.join(', '))}</span>${r.native_grid?'<span>'+esc(r.native_grid.crs+' · pixel size '+r.native_grid.pixel_size.join(' × '))+'</span>':''}</div>`).join('');
}
$('credits-open').onclick=()=>openCredits();$('credits-all').onclick=()=>{creditsFilter=null;creditsLayer=null;renderCredits();};$('credits-search').oninput=renderCredits;
$('methods-open').onclick=()=>{openCredits();$('methods-details').open=true;$('methods-details').scrollIntoView({block:'nearest'});};
document.addEventListener('click',e=>{const source=e.target.closest('[data-source]');if(source)openCredits([source.dataset.source]);const close=e.target.closest('[data-close]');if(close)$(close.dataset.close).close();});

// Plotly 3D: a real camera and fixed physical terrain with dated footprint overlays.
const terrainCamera={eye:{x:.95,y:-1.12,z:.8},up:{x:0,y:0,z:1}};
function plotlyScale(spec){
  if(spec.discrete){let out=[];const n=spec.colours.length;spec.colours.forEach((c,i)=>{out.push([i/n,c],[(i+1)/n,c]);});return out;}
  return spec.colours.map((c,i)=>[i/(spec.colours.length-1),c]);
}
function surfaceTrace(data,z,values,spec,name,opacity=1){return {
  type:'surface',x:data.x,y:data.y,z,surfacecolor:values,colorscale:plotlyScale(spec),cmin:spec.limits[0],cmax:spec.limits[1],
  showscale:false,connectgaps:false,name,opacity,hovertemplate:'Easting offset %{x:.2f} km<br>Northing offset %{y:.2f} km<br>Elevation %{z:.1f} m<extra>'+esc(name)+'</extra>',
  lighting:{ambient:.7,diffuse:.65,specular:.12,roughness:.9},lightposition:{x:-1000,y:500,z:2000},contours:{z:{show:false}}
};}
function mixColour(a,b,f) {
  const rgb=h=>h.replace('#','').match(/../g).map(v=>parseInt(v,16));
  const aa=rgb(a),bb=rgb(b);return '#'+aa.map((v,i)=>Math.round(v*(1-f)+bb[i]*f).toString(16).padStart(2,'0')).join('');
}
function samplePalette(spec,t){
  if(spec.discrete)return spec.colours[Math.min(spec.colours.length-1,Math.floor(t*spec.colours.length))];
  const x=t*(spec.colours.length-1),i=Math.min(spec.colours.length-2,Math.floor(x));
  return mixColour(spec.colours[i],spec.colours[i+1],x-i);
}
function make3DTraces(){
  const s=terrainState,data=s.data,base=data.surface;
  const selected=data.options.find(o=>o.id===$('terrain-colour').value);
  const spec=selected?selected.style:D.elevation_style, values=selected?selected.values:base;
  const showIce=$('terrain-ice').checked,mask=data.ice[years[s.index]],iceColour=colours[years[s.index]];
  // One mesh: colour codes carry the optional ice tint; extra coplanar surfaces waste GPU memory.
  const palette=['#334a59',...Array.from({length:127},(_,i)=>samplePalette(spec,i/126))];
  palette.push(...palette.map(c=>mixColour(c,iceColour,.4)));
  const codes=base.map((row,r)=>row.map((z,c)=>{
    const v=values[r][c];let code=v===null?0:1+Math.round(Math.max(0,Math.min(1,(v-spec.limits[0])/(spec.limits[1]-spec.limits[0])))*126);
    if(!selected&&showIce&&mask&&mask[r][c]==='1')code+=128;
    return code;
  }));
  const trace=surfaceTrace(data,base,codes,{limits:[0,255],colours:palette},selected?selected.title:'2002 surface');
  const traces=[trace];
  if(showIce){
    const line=data.lines[years[s.index]]||[],x=[],y=[],z=[];
    line.forEach(l=>{x.push(...l.x,null);y.push(...l.y,null);z.push(...l.surface.map(v=>v===null?null:v+14),null);});
    if(x.length)traces.push({type:'scatter3d',mode:'lines',x,y,z,line:{color:iceColour,width:5},connectgaps:false,showlegend:false,hoverinfo:'skip',name:'Inventory boundary'});
  }
  return traces;
}
function sceneLayout(){
  const s=terrainState,d=s.data,ex=Number($('terrain-exaggeration').value);
  const dx=Math.max(...d.x)-Math.min(...d.x),dy=Math.max(...d.y)-Math.min(...d.y),span=Math.max(dx,dy,.1);
  const all=d.surface.flat().filter(v=>v!==null),lo=Math.min(...all),hi=Math.max(...all);
  const zr=Math.max(hi-lo,100),range=[lo-30,hi+30];
  return {paper_bgcolor:'#0b1420',plot_bgcolor:'#0b1420',font:{family:'system-ui,sans-serif',color:'#bccbd7',size:10},
    margin:{l:0,r:0,t:0,b:0},showlegend:false,uirevision:s.id,
    scene:{uirevision:s.id,bgcolor:'#0b1420',dragmode:'orbit',camera:s.camera||terrainCamera,
      aspectmode:'manual',aspectratio:{x:dx/span,y:dy/span,z:(zr+60)/1000/span*ex},
      xaxis:{title:{text:'Easting offset (km)'},showbackground:false,gridcolor:'#26384b',zeroline:false},
      yaxis:{title:{text:'Northing offset (km)'},showbackground:false,gridcolor:'#26384b',zeroline:false},
      zaxis:{title:{text:'Elevation (m)'},showbackground:false,gridcolor:'#26384b',zeroline:false,range}},
    annotations:[{text:esc(d.name)+' · 2002 SURFACE',xref:'paper',yref:'paper',x:.03,y:.97,xanchor:'left',showarrow:false,font:{color:'#e8eff5',size:13}}]};
}
async function drawTerrain(){
  const s=terrainState;if(!s||!s.data.available||!window.Plotly)return;
  if(s.drawing){s.pending=true;return;}
  s.drawing=true;
  try{
    const plot=$('terrain-plot');
    if(plot._fullLayout?.scene?.camera)s.camera=JSON.parse(JSON.stringify(plot._fullLayout.scene.camera));
    await Plotly.react(plot,make3DTraces(),sceneLayout(),{responsive:true,plotGlPixelRatio:1,displaylogo:false,scrollZoom:true,modeBarButtonsToRemove:['toImage'],displayModeBar:true});
    const option=s.data.options.find(o=>o.id===$('terrain-colour').value);
    $('terrain-legend').innerHTML=legend(option?option.style:D.elevation_style)+(option?'<p>Grey = no value for this colour layer. The dated ice footprint is outlined so its tint does not alter the data colours.</p>':'<p>The dated footprint is tinted when enabled. Untick it to inspect unblended elevation colours.</p>');
    $('terrain-year').textContent=years[s.index];$('terrain-year-range').value=s.index;
    $('terrain-date-status').textContent=!s.data.ice[years[s.index]]?'No saved outline for this date.':years[s.index]>2014 && (s.id==='island'||!focus.includes(s.data.name))?'Recent non-focus inventories are provisional.':'Mapped inventory date · fixed terrain.';
  }catch(error){s.error=error.message;if(window.Plotly)Plotly.purge($('terrain-plot'));if(terrainState===s)$('terrain-plot').innerHTML='<div class="plot-error">3D rendering could not start. Check that WebGL/hardware acceleration is enabled in your browser.<br>'+esc(String(error.message).slice(0,300))+'</div>';stopTerrain();}
  finally{s.drawing=false;if(s.pending&&terrainState===s){s.pending=false;drawTerrain();}}
}
function stopTerrain(){if(!terrainState)return;terrainState.playing=false;clearTimeout(terrainState.timer);if($('terrain-play'))$('terrain-play').textContent='▶ Play';}
async function stepTerrain(){const s=terrainState;if(!s?.playing)return;await drawTerrain();if(terrainState!==s||!s.playing)return;s.timer=setTimeout(()=>{s.index=(s.index+1)%5;stepTerrain();},1350);}
async function openTerrain(id){
  stopTerrain();island.timeline.pause();if(glacierView)glacierView.timeline.pause();
  const definition=id==='island'?{name:'Heard Island',terrain:'assets/data/island_terrain.js'}:D.glaciers.find(g=>g.id===id);
  if(!definition)return;
  $('terrain-title').textContent=definition.name;$('terrain-plot').innerHTML='<div class="loading">Loading terrain…</div>';
  if(!$('terrain-dialog').open)$('terrain-dialog').showModal();
  const state={id,data:null,index:2,playing:false,drawing:false,camera:null};terrainState=state;
  try{
    await Promise.all([window.Plotly?Promise.resolve():loadScript(D.plotly_url),H.terrain[id]?Promise.resolve():loadScript(definition.terrain)]);
    if(terrainState!==state||!$('terrain-dialog').open)return;
    const data=H.terrain[id];state.data=data;
    if(!data?.available){$('terrain-plot').innerHTML='<div class="plot-error"><h3>Terrain is unavailable for this view</h3>'+esc(data?.reason||'No terrain asset was produced.')+'</div>';return;}
    $('terrain-colour').innerHTML='<option value="elevation">Elevation</option>'+data.options.map(o=>`<option value="${esc(o.id)}">${esc(o.title)}</option>`).join('');
    $('terrain-exaggeration').value=1;$('terrain-exaggeration-label').textContent='1×';$('terrain-ice').checked=true;
    $('terrain-timeline').innerHTML='<div class="timeline-title">RETREAT · MAPPED DATES</div><div class="timeline-main"><button id="terrain-play">▶ Play</button><input id="terrain-year-range" type="range" min="0" max="4" value="2" step="1" aria-label="3D inventory date"><output id="terrain-year" class="timeline-year">2014</output></div><div class="timeline-dates">'+years.map(y=>`<span>${y}</span>`).join('')+'</div>';
    $('terrain-grid-note').textContent=`Display grid: ${data.shape[1]} × ${data.shape[0]} cells; approximately ${Math.round(data.display_cell_m[0])} m spacing. Original analytical rasters are unchanged.`;
    $('terrain-play').onclick=()=>{if(state.playing)stopTerrain();else{state.playing=true;$('terrain-play').textContent='Ⅱ Pause';stepTerrain();}};
    $('terrain-year-range').oninput=e=>{stopTerrain();state.index=Number(e.target.value);drawTerrain();};
    $('terrain-colour').onchange=()=>drawTerrain();$('terrain-ice').onchange=()=>drawTerrain();
    $('terrain-exaggeration').oninput=e=>{$('terrain-exaggeration-label').textContent=e.target.value+'×';drawTerrain();};
    $('terrain-sources').onclick=()=>openCredits([...new Set([...data.sources,...data.options.flatMap(o=>o.sources)])]);
    $('terrain-reset').onclick=()=>{state.camera=JSON.parse(JSON.stringify(terrainCamera));Plotly.relayout($('terrain-plot'),{'scene.camera':state.camera});};
    $('terrain-plot').replaceChildren();await drawTerrain();
  }catch(error){if(terrainState===state)$('terrain-plot').innerHTML='<div class="plot-error">'+esc(error.message)+'</div>';}
}
$('island-3d').onclick=()=>openTerrain('island');
$('terrain-dialog').addEventListener('close',()=>{stopTerrain();if(window.Plotly)Plotly.purge($('terrain-plot'));terrainState=null;$('terrain-plot').replaceChildren();});
window.addEventListener('resize',resizeViews);
// Expose a small inspection surface for diagnostics and automated interaction checks.
H.app={openGlacier,openTerrain,openCredits,get island(){return island;},get glacier(){return glacierView;},get terrain(){return terrainState;}};
})();

'''


## Export the application

Pinned Leaflet and Plotly assets are cached for portability. If their download is unavailable, the HTML uses the specified online versions. Online imagery still requires internet. The old overview filename redirects to the rebuilt explorer.

In [ ]:
## Export the browser application and its source register ##


def vendor_asset(filename, url, minimum_bytes):
    """Cache a pinned browser library locally; retain a CDN fallback if offline at build time."""
    path = asset_dir / "vendor" / filename
    if path.is_file() and path.stat().st_size >= minimum_bytes:
        return path.relative_to(explorer_dir).as_posix()
    try:
        request = Request(url, headers = {"User-Agent": "Heard-Island-project-map/1.0"})
        with urlopen(request, timeout = 25) as response:
            content = response.read()
        if len(content) < minimum_bytes:
            raise ValueError("Library response was unexpectedly small.")
        path.write_bytes(content)
        return path.relative_to(explorer_dir).as_posix()
    except Exception as exc:
        build_issues.append({"item": filename, "detail": "Online fallback: " + str(exc)})
        return url


leaflet_js = vendor_asset("leaflet-1.9.4.js", "https://unpkg.com/leaflet@1.9.4/dist/leaflet.js", 100000)
leaflet_css = vendor_asset("leaflet-1.9.4.css", "https://unpkg.com/leaflet@1.9.4/dist/leaflet.css", 10000)
plotly_js = vendor_asset("plotly-3.1.0.min.js", "https://cdn.plot.ly/plotly-3.1.0.min.js", 1000000)
# These are software licences, separate from the scientific dataset licences.
vendor_asset("leaflet-LICENSE.txt", "https://unpkg.com/leaflet@1.9.4/LICENSE", 500)
vendor_asset("plotly-LICENSE.txt", "https://unpkg.com/plotly.js@3.1.0/LICENSE", 500)

for record in layer_catalogue:
    record["available"] = project_path(record["path"]).is_file()
    record["overview"] = record["id"].startswith("ice_") or any(l["id"] == record["id"] for l in overview_layers)
    if record["available"]:
        p = project_path(record["path"])
        record["file_metadata"] = {"bytes": p.stat().st_size, "modified_utc": datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).isoformat()}

# Warn explicitly about source files that exist but could not be displayed.
for record in layer_catalogue:
    if not record["available"]:
        build_issues.append({"item": record["id"], "detail": "Source file not found: " + record["path"]})

for filename, content in {"source_register.json": source_register, "layer_catalogue.json": layer_catalogue,
        "processing_provenance.json": provenance_register, "build_issues.json": build_issues}.items():
    write_json(explorer_dir / filename, content)

overview_payload = {"build_id": build_id, "bounds": geographic_bounds(island_domain),
    "history": island_history, "glaciers": selector, "layers": overview_layers,
    "year_colours": year_colours, "glacier_colours": map_colours, "rock_colours": rock_colours,
    "tooltip_labels": tooltip_labels, "sources": source_register, "catalogue": layer_catalogue,
    "technical_references": technical_references, "inventory_counts": inventory_counts,
    "inventory_sources": {str(y): records[f"ice_{y}"]["sources"] for y in years},
    "imagery_credit": imagery_credit, "plotly_url": plotly_js, "elevation_style": spec_lookup["surface"]}
write_data_script(asset_dir / "data" / "overview.js", "window.HI.overview", overview_payload)

# APP_HTML, APP_CSS and APP_JS are defined in the three cells immediately above.
html = APP_HTML.replace("@@LEAFLET_CSS@@", leaflet_css).replace("@@LEAFLET_JS@@", leaflet_js).replace("@@BUILD@@", build_id)
map_path = explorer_dir / "Heard_Island_explorer.html"
map_path.write_text(html, encoding = "utf-8")
(asset_dir / "explorer.css").write_text(APP_CSS, encoding = "utf-8")
(asset_dir / "explorer.js").write_text(APP_JS, encoding = "utf-8")

# Preserve the old entry point as a lightweight redirect, so existing bookmarks still open.
(explorer_dir / "Heard_Island_overview.html").write_text(
    '<!doctype html><meta charset="utf-8"><meta http-equiv="refresh" content="0;url=Heard_Island_explorer.html">'
    '<title>Heard Island explorer</title><a href="Heard_Island_explorer.html">Open the Heard Island explorer</a>', encoding = "utf-8")
print(f"Saved: {map_path}")

## Validate and package the export

Check the requested overview exclusions, inclusion of all available recent inventories, source associations and referenced assets. The portable ZIP contains the current map assets and provenance records, without superseded raster-cache revisions.

In [ ]:
## Check the export and make a portable copy ##

# These checks catch missing browser assets and accidental scope regressions before display.
assert len(dashboard_manifest) == len(glacier_catalogue)
assert len({r["id"] for r in layer_catalogue}) == len(layer_catalogue)
assert not any(l["id"] in excluded_overview or l["tab"] == "Field evidence" for l in overview_layers)
assert all(source_id in source_register for record in layer_catalogue for source_id in record["sources"])
assert all((explorer_dir / g["data"]).is_file() and (explorer_dir / g["terrain"]).is_file() for g in selector)
assert all((explorer_dir / l["image"]).is_file() for l in overview_layers if l["type"] == "raster")
if "coastline" in map_vectors:
    assert not map_vectors["coastline"].geom_type.isin(["Polygon", "MultiPolygon"]).any()
for year in [2019, 2026]:
    included = {f["properties"].get("RGIId") for f in island_history if f["properties"]["year"] == year}
    assert set(outlines.loc[outlines.year.eq(year), "RGIId"]) <= included

software = {"Python": sys.version.split()[0]}
for package in ["numpy", "pandas", "geopandas", "rasterio", "shapely", "matplotlib", "Pillow", "IPython"]:
    software[package] = importlib.metadata.version(package)
manifest = {"build_utc": build_id, "software": software, "browser_software": {"Leaflet": "1.9.4", "Plotly.js": "3.1.0"},
    "glaciers": len(glacier_catalogue), "inventory_counts": inventory_counts, "overview_layers": len(overview_layers),
    "dashboard_layers": sum(g["layers"] for g in dashboard_manifest), "terrain_scenes": terrain_manifest,
    "issue_count": len(build_issues), "sources_requiring_bibliographic_check": [s["id"] for s in source_register.values() if s["citation_status"] != "verified"],
    "notebook_display": "Small iframe served from an export-only loopback HTTP server; no image data embedded in output",
    "portable_export": "Open Heard_Island_explorer.html with its assets folder. Python is not required. Online imagery requires internet."}
write_json(explorer_dir / "build_manifest.json", manifest)

readme = """HEARD ISLAND EXPLORER

Open Heard_Island_explorer.html in a modern browser. Keep the assets folder beside it.
The export runs without Python or a local server. Online imagery needs internet.
If a pinned browser library could not be cached during the build, it also needs internet.

BOOK EMBED: copy this complete folder into the book's static assets and embed
Heard_Island_explorer.html in an iframe (width 100%, height about 820px).
Do not embed a localhost preview address in the published book.

The scientific GeoTIFFs are not included in this display export. PNGs and 3D grids
are bounded visual copies, not replacements for the analytical datasets.
Read source_register.json, layer_catalogue.json and processing_provenance.json
for citations, dataset lineage and processing settings. build_issues.json lists
missing or failed optional inputs; no replacement science data are invented.

2019 and 2026 non-focus inventories are provisional context. Absent saved outlines
are left absent. 3D retreat animates mapped footprints on fixed terrain, not historical
surface elevations or thickness. Modelled bed gaps beneath ice remain gaps.
"""
(explorer_dir / "README.txt").write_text(readme, encoding = "utf-8")

# Include only PNGs referenced by current data scripts, not obsolete cached revisions.
referenced_images = {l["image"] for l in overview_layers if l["type"] == "raster"}
for glacier in selector:
    script = (explorer_dir / glacier["data"]).read_text(encoding = "utf-8")
    payload = json.loads(script.split(" = ", 1)[1].rstrip(";\n"))
    referenced_images.update(l["image"] for l in payload["layers"] if l["type"] == "raster")
    assert all((explorer_dir / l["image"]).is_file() for l in payload["layers"] if l["type"] == "raster")
    assert len(payload["layers"]) == len({l["id"] for l in payload["layers"]})
if MAKE_PORTABLE_ZIP:
    zip_path = map_data_dir / "Heard_Island_explorer_portable.zip"
    with zipfile.ZipFile(zip_path, "w", compression = zipfile.ZIP_DEFLATED, compresslevel = 6) as archive:
        for path in explorer_dir.rglob("*"):
            if not path.is_file() or path.suffix.lower() == ".log":
                continue
            relative = path.relative_to(explorer_dir).as_posix()
            if relative.startswith("raster_cache/") or relative.startswith("assets/rasters/") and relative not in referenced_images:
                continue
            # Exclude superseded HTML builds while retaining this export and its redirect.
            if path.suffix.lower() == ".html" and path.name not in ["Heard_Island_explorer.html", "Heard_Island_overview.html"]:
                continue
            archive.write(path, "Heard_Island_explorer/" + relative)
    print(f"Portable folder archive: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
print(f"Checks passed: {len(glacier_catalogue)} dashboards; {len(referenced_images)} referenced images.")
if build_issues:
    print(f"{len(build_issues)} missing/optional-input notes saved to build_issues.json.")
    display(pd.DataFrame(build_issues).head(12))
print("Source records with unresolved bibliographic details:", ", ".join(manifest["sources_requiring_bibliographic_check"]))

## Preview the map in the notebook

This cell starts a small **export-only** server bound to `127.0.0.1`, then displays an iframe. The output contains a URL, not embedded raster data. Rerunning this cell shuts down the previous preview server.

Technical reference: [IPython IFrame](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.IFrame).

In [ ]:
## Display the map without embedding its assets in notebook output ##

# Re-running this cell replaces the previous preview server cleanly.
# It serves only the exported map folder, on this computer, without listing directories.
if "preview_server" in globals():
    preview_server.shutdown()
    preview_server.server_close()


class ExplorerPreviewHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass

    def end_headers(self):
        self.send_header("Cache-Control", "no-store")
        super().end_headers()

    def list_directory(self, path):
        self.send_error(403, "Open Heard_Island_explorer.html directly.")
        return None


preview_server = ThreadingHTTPServer(("127.0.0.1", 0), partial(ExplorerPreviewHandler, directory = str(explorer_dir)))
preview_thread = Thread(target = preview_server.serve_forever, daemon = True)
preview_thread.start()
preview_url = f"http://127.0.0.1:{preview_server.server_port}/Heard_Island_explorer.html?v={build_id}"
display(HTML('<p><a href="' + escape(preview_url, quote = True) + '" target="_blank" rel="noopener">'
    'Open the full-window explorer ↗</a></p>'))
# IFrame output is a small URL, rather than many megabytes of embedded raster images.
display(IFrame(src = preview_url, width = "100%", height = 820, extras = ['allow="fullscreen"', "allowfullscreen"]))
print(f"Standalone export: {map_path}")
print("The notebook preview needs this kernel running. The exported HTML and assets do not.")

## Use the export in the book or presentation

- Open `Heard_Island_explorer.html` for a full-window map. Keep `assets/` beside it.
- Share or extract `Heard_Island_explorer_portable.zip` to move the complete browser export.
- For the book, copy the exported folder into the book's static assets and embed its relative HTML path in an iframe. Do not publish the notebook's localhost preview address.
- Review `build_issues.json` for any unavailable optional inputs and `source_register.json` for unresolved bibliographic details before the final submission.

The main map uses all available recent outlines. The saved inventory may contain fewer than 23 glaciers in a given year; the map does not invent missing geometry. Use the area tables and outline status together when interpreting change.